# Dataset NetFlow e PFS generale: versione autonoma
Questo notebook contiene il catalogo PFS di 29 coordinate, conversioni, politica semantica, audit e test. I domini sono quattro NetFlow-v2, CIC2017, CIC2018 e UGR16. Le due rappresentazioni CSE-CIC-IDS2018 condividono la stessa raccolta originaria.

Le versioni packet15, B29 e i rebuild storici sono rimossi dalla versione operativa. I CSV originali vengono preservati. Il PFS non recupera 29 misure native dove i CSV non le conservano.

La modalità predefinita `report` mostra risultati già ottenuti. Per validare questa riorganizzazione usare `test`; soltanto dopo i test, `audit` permette una nuova verifica campionaria. Nessuna modalità avvia training.


In [ ]:
from pathlib import Path
import os
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import HTML, display

# Point to the CSV parent folder when notebooks and datasets are stored separately.
NOTEBOOK_DIR = Path('.').resolve()
ROOT = Path(os.environ.get('NIDS_PROJECT_ROOT', str(NOTEBOOK_DIR))).resolve()
NOTEBOOK_NAMES = ('Autoencoder_2.0.ipynb', 'Autoencoder_NetFlow_Multidomain.ipynb',
                  'Dataset.ipynb', 'Dataset_2018.ipynb', 'Dataset_NetFlow_Preparation.ipynb')
def show_table(rows, title):
    print(title)
    display(pd.DataFrame(rows))
DATASET_PFS_MODE = 'report'  # 'report', 'test', 'audit'


## Codice e contratti incorporati
Le celle seguenti conservano moduli separati in memoria per evitare collisioni fra funzioni. Tutto il sorgente e tutti i JSON sono nel notebook; non vengono creati o letti script del progetto. I percorsi virtuali servono soltanto a identificare e verificare il codice. NumPy, pandas, TensorFlow e le altre librerie rimangono dipendenze dell'ambiente Python.


In [ ]:
NOTEBOOK_RUNTIME_SOURCE = r'''import builtins
import hashlib
import io
import json
import linecache
import os
import pathlib
import sys
import types
from pathlib import Path as _RealPath

_PFS_TEXT = {}
_PFS_HASHES = {}
_PFS_VIRTUAL = {}
_PFS_MODULES = {}
_REAL_IMPORT = builtins.__import__

def _pfs_put(name, text, expected):
    data = text.encode('utf-8')
    if hashlib.sha256(data).hexdigest() != expected:
        raise ValueError('Contenuto incorporato alterato: ' + name)
    _PFS_TEXT[name] = text
    _PFS_HASHES[name] = expected

def _pfs_virtual_data(path):
    return _PFS_VIRTUAL.get(str(_RealPath(path).resolve()).casefold())

class _NotebookPath(type(_RealPath())):
    """Read-only embedded sources; ordinary datasets/artifacts use normal file I/O."""
    def exists(self):
        return _pfs_virtual_data(self) is not None or super().exists()
    def is_file(self):
        return _pfs_virtual_data(self) is not None or super().is_file()
    def open(self, mode='r', buffering=-1, encoding=None, errors=None, newline=None):
        data = _pfs_virtual_data(self)
        if data is None:
            return super().open(mode, buffering, encoding, errors, newline)
        if mode not in ('r', 'rb', 'rt'):
            raise PermissionError('I sorgenti incorporati sono di sola lettura: ' + str(self))
        raw = io.BytesIO(data)
        return raw if 'b' in mode else io.TextIOWrapper(raw, encoding=encoding or 'utf-8', errors=errors, newline=newline)
    def read_bytes(self):
        data = _pfs_virtual_data(self)
        return super().read_bytes() if data is None else data
    def read_text(self, encoding=None, errors=None):
        data = _pfs_virtual_data(self)
        return super().read_text(encoding=encoding, errors=errors) if data is None else data.decode(encoding or 'utf-8', errors or 'strict')

_PATHLIB = types.ModuleType('pathlib')
_PATHLIB.__dict__.update(pathlib.__dict__)
_PATHLIB.Path = _NotebookPath

def _pfs_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name == 'pathlib' and level == 0:
        return _PATHLIB
    return _REAL_IMPORT(name, globals, locals, fromlist, level)

def _pfs_initialize(root, runtime_text):
    _PFS_TEXT['notebook_runtime.py'] = runtime_text
    _PFS_HASHES['notebook_runtime.py'] = hashlib.sha256(runtime_text.encode()).hexdigest()
    _PFS_VIRTUAL.clear()
    sys.modules.pop('notebook_nf_roles', None)
    for relative, text in _PFS_TEXT.items():
        if hashlib.sha256(text.encode()).hexdigest() != _PFS_HASHES[relative]:
            raise ValueError('Hash incorporato non coerente: ' + relative)
        _PFS_VIRTUAL[str((_RealPath(root)/relative).resolve()).casefold()] = text.encode()
    # These packages live entirely in memory: no project .py file is imported.
    for package in ('flow_pipeline', 'models', 'tools'):
        for name in list(sys.modules):
            if name == package or name.startswith(package+'.'):
                del sys.modules[name]
        module = types.ModuleType(package)
        module.__path__ = []
        sys.modules[package] = module
    for relative, text in _PFS_TEXT.items():
        if not relative.endswith('.py') or relative == 'notebook_runtime.py':
            continue
        name = relative[:-3].replace('/', '.')
        module = types.ModuleType(name)
        module.__file__ = str(_RealPath(root)/relative)
        module.__package__ = name.rpartition('.')[0]
        module.__dict__['__builtins__'] = dict(vars(builtins), __import__=_pfs_import)
        sys.modules[name] = module
        _PFS_MODULES[name] = module
        if module.__package__:
            setattr(sys.modules[module.__package__], name.rsplit('.',1)[-1], module)
        # Keras/inspect can retrieve the actual embedded source without reading .py files.
        linecache.cache[module.__file__] = (len(text), None, text.splitlines(keepends=True), module.__file__)
        exec(compile(text, module.__file__, 'exec'), module.__dict__)
    return _NotebookPath(root)

def _pfs_check_bundle(notebook_names):
    """User-run syntax/hash checks, including all five notebooks when available."""
    for relative, text in _PFS_TEXT.items():
        if relative.endswith('.py'):
            compile(text, relative, 'exec')
        else:
            json.loads(text)
    checked = []
    for name in notebook_names:
        path = _RealPath(NOTEBOOK_DIR)/name
        if not path.exists():
            continue
        nb = json.loads(path.read_text(encoding='utf-8'))
        for index, cell in enumerate(nb['cells']):
            if cell['cell_type'] == 'code':
                compile(''.join(cell['source']), name+':cell'+str(index+1), 'exec')
        checked.append(name)
    return {'embedded_resources': len(_PFS_TEXT), 'notebooks_syntax_checked': checked, 'external_project_modules_required': False}'''
exec(compile(NOTEBOOK_RUNTIME_SOURCE, '<notebook-runtime>', 'exec'))


In [ ]:
# Contratto incorporato: feature_contracts/portable_general29_v1.json
_pfs_put('feature_contracts/portable_general29_v1.json', r'''{
  "version": "portable_general29_v1",
  "feature_count": 29,
  "scope": "exported_flow_record",
  "primary_profile": "common9",
  "quality_codes": {"missing": 0, "proxy": 1, "native_or_exact_derived": 2},
  "profiles": {
    "common9": ["packets", "duration_ms", "packet_rate_s", "tcp_fin", "tcp_syn", "tcp_rst", "tcp_psh", "tcp_ack", "tcp_urg"],
    "common12_bytes_proxy": ["ip_bytes", "packets", "duration_ms", "mean_ip_packet_bytes", "byte_rate_s", "packet_rate_s", "tcp_fin", "tcp_syn", "tcp_rst", "tcp_psh", "tcp_ack", "tcp_urg"],
    "extended29_masked": "all"
  },
  "features": [
    {"name": "ip_bytes", "group": "volume", "kind": "continuous", "unit": "bytes", "definition": "IP octets represented by the exported record; sum both directions when present"},
    {"name": "packets", "group": "volume", "kind": "continuous", "unit": "packets", "definition": "Packets represented by the exported record; sum both directions when present"},
    {"name": "duration_ms", "group": "time", "kind": "continuous", "unit": "ms", "definition": "Total duration of the exported record; not directional IAT Total"},
    {"name": "mean_ip_packet_bytes", "group": "size", "kind": "continuous", "unit": "bytes/packet", "definition": "ip_bytes / packets; missing when packets <= 0"},
    {"name": "byte_rate_s", "group": "rate", "kind": "continuous", "unit": "bytes/s", "definition": "1000 * ip_bytes / duration_ms; missing at zero duration; not native nProbe SECOND_BYTES"},
    {"name": "packet_rate_s", "group": "rate", "kind": "continuous", "unit": "packets/s", "definition": "1000 * packets / duration_ms; missing at zero duration"},
    {"name": "tcp_fin", "group": "tcp_flags", "kind": "binary", "unit": "indicator", "definition": "FIN observed at least once; bit presence, never packet count"},
    {"name": "tcp_syn", "group": "tcp_flags", "kind": "binary", "unit": "indicator", "definition": "SYN observed at least once"},
    {"name": "tcp_rst", "group": "tcp_flags", "kind": "binary", "unit": "indicator", "definition": "RST observed at least once"},
    {"name": "tcp_psh", "group": "tcp_flags", "kind": "binary", "unit": "indicator", "definition": "PSH observed at least once"},
    {"name": "tcp_ack", "group": "tcp_flags", "kind": "binary", "unit": "indicator", "definition": "ACK observed at least once"},
    {"name": "tcp_urg", "group": "tcp_flags", "kind": "binary", "unit": "indicator", "definition": "URG observed at least once"},
    {"name": "tcp_ece", "group": "tcp_flags", "kind": "binary", "unit": "indicator", "definition": "ECE observed at least once; missing in six-character nfdump flags"},
    {"name": "tcp_cwr", "group": "tcp_flags", "kind": "binary", "unit": "indicator", "definition": "CWR observed at least once; missing in six-character nfdump flags"},
    {"name": "protocol_tcp", "group": "protocol", "kind": "binary", "unit": "indicator", "definition": "Observed IP protocol is TCP (6); missing if protocol absent"},
    {"name": "protocol_udp", "group": "protocol", "kind": "binary", "unit": "indicator", "definition": "Observed IP protocol is UDP (17)"},
    {"name": "protocol_icmp", "group": "protocol", "kind": "binary", "unit": "indicator", "definition": "Observed IP protocol is ICMP (1) or ICMPv6 (58)"},
    {"name": "src_port_system", "group": "src_service", "kind": "binary", "unit": "indicator", "definition": "Observed TCP/UDP source port in [0,1023]"},
    {"name": "src_port_user", "group": "src_service", "kind": "binary", "unit": "indicator", "definition": "Observed TCP/UDP source port in [1024,49151]"},
    {"name": "src_port_dynamic", "group": "src_service", "kind": "binary", "unit": "indicator", "definition": "Observed TCP/UDP source port in [49152,65535]"},
    {"name": "dst_port_system", "group": "dst_service", "kind": "binary", "unit": "indicator", "definition": "Observed TCP/UDP destination port in [0,1023]"},
    {"name": "dst_port_user", "group": "dst_service", "kind": "binary", "unit": "indicator", "definition": "Observed TCP/UDP destination port in [1024,49151]"},
    {"name": "dst_port_dynamic", "group": "dst_service", "kind": "binary", "unit": "indicator", "definition": "Observed TCP/UDP destination port in [49152,65535]"},
    {"name": "ip_packet_min_bytes", "group": "size", "kind": "continuous", "unit": "bytes", "definition": "Minimum IP packet length; CIC payload minimum is only a proxy"},
    {"name": "ip_packet_max_bytes", "group": "size", "kind": "continuous", "unit": "bytes", "definition": "Maximum IP packet length; CIC payload maximum is only a proxy"},
    {"name": "reverse_packet_share", "group": "direction", "kind": "bounded", "unit": "fraction", "definition": "Backward packets / total packets; missing on unidirectional records"},
    {"name": "reverse_byte_share", "group": "direction", "kind": "bounded", "unit": "fraction", "definition": "Backward IP octets / total IP octets; CIC estimate remains proxy"},
    {"name": "tcp_window_max_fwd", "group": "window", "kind": "continuous", "unit": "bytes", "definition": "Maximum unscaled forward TCP window; CIC initial window is only a proxy"},
    {"name": "tcp_window_max_bwd", "group": "window", "kind": "continuous", "unit": "bytes", "definition": "Maximum unscaled backward TCP window; CIC initial window is only a proxy"}
  ],
  "cic_bytes_policy": "optional_ipv4_no_options_proxy: payload + transport_headers + 20 * packets",
  "zero_duration_policy": "retain zero duration; rates missing, no epsilon denominator",
  "absent_feature_policy": "NaN + explicit per-cell quality; source-only imputation after conversion",
  "preprocessing": "continuous log1p + source-benign robust scaling; binary and bounded retain original scale; identical clipping in training and scoring",
  "fitting": "source benign TRAIN only; no target fitting; calibration on disjoint source benign CALIBRATION",
  "identifiers_as_input": false,
  "labels_as_input": false,
  "limits": [
    "29 semantic coordinates in the catalog, not 29 observed or independent measurements on every exporter",
    "Port and protocol indicators are encodings of categorical primitives, not independent measurements",
    "Record directionality, flow timeouts and segmentation remain exporter dependent",
    "The common9 profile excludes byte proxies, ports, protocol, windows and unavailable ECE/CWR",
    "The common12 profile deliberately accepts the CIC byte proxy and its dependent quantities",
    "CIC historical flag extraction bugs require per-file diagnostics; column presence does not certify correct values",
    "Importance is motivated by flow behavior; predictive importance is not established before source-only experiments"
  ],
  "sources": [
    "https://www.ntop.org/guides/nprobe/flow_information_elements.html",
    "https://github.com/ahlashkari/CICFlowMeter/blob/master/src/main/java/cic/cs/unb/ca/jnetpcap/BasicFlow.java",
    "https://github.com/phaag/nfdump"
  ]
}
''', '44bd4956cd21181534f70a5d2cde0a84096183b7048c54c8b27d4c7fa047a238')


In [ ]:
# Contratto incorporato: feature_contracts/portable_general_semantics_v1.json
_pfs_put('feature_contracts/portable_general_semantics_v1.json', r'''{
  "version": "portable_general_semantics_v1",
  "catalog": "portable_general29_v1",
  "scope": "current_local_csv_inventory_only",
  "notice": "Eligibility is a conservative review decision, not a certification of physical accuracy, session equivalence or predictive importance. Mapping quality is preserved separately.",
  "quality_codes": {
    "0": "missing, invalid, or excluded by a semantic rule; inspect exclusion reasons",
    "1": "declared proxy without a known semantic exclusion",
    "2": "direct or exact derivation without a known semantic exclusion"
  },
  "domains": {
    "NF-CSE-CIC-IDS2018-v2": "netflow_v2",
    "NF-UNSW-NB15-v2": "netflow_v2",
    "NF-ToN-IoT-v2": "netflow_v2",
    "NF-BoT-IoT-v2": "netflow_v2",
    "CIC2017": "cicflowmeter",
    "CIC2018": "cicflowmeter",
    "UGR16": "ugr16"
  },
  "rules": [
    {
      "id": "nf_v2_duration_unresolved",
      "domains": ["NF-CSE-CIC-IDS2018-v2", "NF-UNSW-NB15-v2", "NF-ToN-IoT-v2", "NF-BoT-IoT-v2"],
      "features": ["duration_ms", "byte_rate_s", "packet_rate_s"],
      "evidence_level": "local_sample_signal_conservative_exclusion",
      "reason": "I campioni di tutti i quattro NF mostrano durata totale prossima a 2^32/1000 meno il massimo dei tempi direzionali. Causa e correzione non dimostrate; escluse durata e derivate dall'intero profilo del file, compresi gli zeri. DURATION_IN/OUT non sostituiscono la durata biflow.",
      "evidence": ["local_nf_duration_samples_20261003"]
    },
    {
      "id": "cic2017_historical_flags",
      "domains": ["CIC2017"],
      "features": ["tcp_fin", "tcp_syn", "tcp_rst", "tcp_psh", "tcp_ack", "tcp_urg", "tcp_ece", "tcp_cwr"],
      "evidence_level": "published_dataset_defect_and_unknown_local_extraction_provenance",
      "reason": "Lo studio sui CSV storici CICIDS2017 documenta conteggi TCP aggiornati solo per il primo pacchetto. Il PFS richiede presenza nell'intero flusso. La versione di estrazione locale non e' attestata; raw conservati, flag esclusi. Nessuna unione dei conteggi direzionali o scambio automatico di colonne.",
      "evidence": ["cic_historical_flags_wtmc2021"]
    },
    {
      "id": "cic2018_historical_flags_unverified",
      "domains": ["CIC2018"],
      "features": ["tcp_fin", "tcp_syn", "tcp_rst", "tcp_psh", "tcp_ack", "tcp_urg", "tcp_ece", "tcp_cwr"],
      "evidence_level": "historical_tool_risk_and_local_inconsistency_conservative_exclusion",
      "reason": "Lo stesso difetto puo' interessare dataset prodotti con CICFlowMeter-V3 prima della correzione del 2019. I campioni CIC2018 locali hanno conteggi binari e PSH/URG discordanti dalle direzioni; non provano da soli il difetto. Provenienza non attestata: flag esclusi in via conservativa, senza dichiarare dimostrato il bug per ogni riga.",
      "evidence": ["cic_historical_flags_wtmc2021", "local_cic_flag_samples_20261003"]
    }
  ],
  "row_rules": [
    {
      "id": "nf_zero_ip_minimum",
      "domains": ["NF-CSE-CIC-IDS2018-v2", "NF-UNSW-NB15-v2", "NF-ToN-IoT-v2", "NF-BoT-IoT-v2"],
      "features": ["ip_packet_min_bytes"],
      "reason": "Minimo IP uguale a zero con pacchetti positivi: valore escluso per riga, possibile sentinella non ricostruita."
    }
  ],
  "dependencies": {
    "mean_ip_packet_bytes": ["ip_bytes", "packets"],
    "byte_rate_s": ["ip_bytes", "duration_ms"],
    "packet_rate_s": ["packets", "duration_ms"],
    "reverse_byte_share": ["ip_bytes"],
    "reverse_packet_share": ["packets"]
  },
  "exploratory_common_volume": ["packets", "ip_bytes", "mean_ip_packet_bytes"],
  "exploratory_common_volume_notice": "Three coordinates from two counters. CIC IP bytes are estimated, so this is not an exact universal core or a selected training profile.",
  "retired_training_profiles": ["common7_no_time", "common9_no_time_bytes_proxy"],
  "accepted_training_profile": null,
  "evidence": [
    {
      "id": "cic_historical_flags_wtmc2021",
      "kind": "primary_research_documentation",
      "url": "https://intrusion-detection.distrinet-research.be/WTMC2021/extended_doc.html",
      "section": "TCP flag count",
      "accessed": "2026-10-03"
    },
    {
      "id": "local_nf_duration_samples_20261003",
      "kind": "saved_user_run_bounded_samples",
      "report": "outputs/pfsa/7bd2efc1b319e7cf/report.json",
      "check": "nf_near_unsigned32_microseconds_pattern",
      "counts_by_domain": {
        "NF-CSE-CIC-IDS2018-v2": 521,
        "NF-UNSW-NB15-v2": 199,
        "NF-ToN-IoT-v2": 846,
        "NF-BoT-IoT-v2": 3667
      },
      "notice": "Four byte windows of at most 1000 candidate lines per file; not a full scan or proof of an overflow repair."
    },
    {
      "id": "local_cic_flag_samples_20261003",
      "kind": "saved_user_run_bounded_samples",
      "report": "outputs/pfsa/7bd2efc1b319e7cf/report.json",
      "checks": ["cic_psh_directional_presence_vs_total", "cic_urg_directional_presence_vs_total"],
      "notice": "Local inconsistencies motivate review; they do not identify the historical tool build or recover cumulative flags."
    }
  ]
}
''', '97a8da02b80ea9d6e77ce5e1e82b9f6f42312e1198b8fcfc4ec8f6288c46536b')


In [ ]:
# Contratto incorporato: feature_contracts/portable_general_masked_pilot_v1.json
_pfs_put('feature_contracts/portable_general_masked_pilot_v1.json', r'''{
  "version": "portable_general_masked_pilot_v1",
  "catalog": "portable_general29_v1",
  "semantic_policy": "portable_general_semantics_v1",
  "status": "exploratory_pilot_not_validated_detector",
  "profiles": {
    "volume3_proxy": ["ip_bytes", "packets", "mean_ip_packet_bytes"],
    "semantic29_masked_proxy": "all"
  },
  "primary_profile": "semantic29_masked_proxy",
  "primary_threshold_policy": "balanced_p99",
  "input_channels": ["source_scaled_value", "observed_mask", "proxy_mask"],
  "proxy_error_weight": 0.5,
  "minimum_observed_group_weight": 0.5,
  "coverage_comparison_tolerance": 0.0000001,
  "minimum_scored_fraction_per_test_class_for_gate": 0.95,
  "input_view": "audited_original_csvs",
  "fitting": "Source benign TRAIN only; early stopping on source benign STOP; thresholds on disjoint source benign CALIBRATION. No target fitting.",
  "coverage": "Eligibility applied before every fit and score; rows below minimum group coverage are excluded from fit/score and counted by class on test.",
  "comparison": "Report both each-profile coverage and metrics on the intersection of scored holdout rows.",
  "limitations": [
    "29 coordinates in the model output are not 29 observed, independent, native measurements in every record.",
    "The volume comparator contains three coordinates from two counters and accepts the CIC byte estimate.",
    "The extended profile also accepts declared payload-size, initial-window and byte-share proxies; a separate proxy channel and lower loss weight do not make them exact.",
    "Availability patterns can reveal the exporter; measured LODO performance is required before recommending deployment.",
    "Fixed raw prefixes and grouped-record splits do not certify time/session/host disjointness or full-dataset performance.",
    "The current semantic audit remains an information audit, not an accepted training profile; this separate contract authorizes only an explicitly requested exploratory pilot."
  ]
}
''', 'c38c123cd9b51dc0acda0f2800f8674c9242b9a5f6082dd8daf5233173132ba6')


In [ ]:
# Contratto incorporato: feature_contracts/netflow_v2_43.json
_pfs_put('feature_contracts/netflow_v2_43.json', r'''{
  "contract_name": "netflow_v2_standard_43_portable_v1",
  "contract_version": 1,
  "status": "frozen_before_data_download",
  "source": {
    "paper": "Towards a Standard Feature Set for Network Intrusion Detection System Datasets",
    "paper_url": "https://arxiv.org/abs/2101.11315",
    "dataset_url": "https://www.cyber.uq.edu.au/project/machine-learning-based-nids-datasets",
    "extractor": "nProbe",
    "flow_format": "NetFlow v9"
  },
  "expected_dataset_names": [
    "NF-CSE-CIC-IDS2018-v2",
    "NF-UNSW-NB15-v2",
    "NF-ToN-IoT-v2",
    "NF-BoT-IoT-v2",
    "NF-UQ-NIDS-v2"
  ],
  "label_columns": [
    "Label",
    "Attack",
    "Dataset"
  ],
  "feature_sets": {
    "standard_43": [
      "IPV4_SRC_ADDR",
      "L4_SRC_PORT",
      "IPV4_DST_ADDR",
      "L4_DST_PORT",
      "PROTOCOL",
      "L7_PROTO",
      "IN_BYTES",
      "IN_PKTS",
      "OUT_BYTES",
      "OUT_PKTS",
      "TCP_FLAGS",
      "CLIENT_TCP_FLAGS",
      "SERVER_TCP_FLAGS",
      "FLOW_DURATION_MILLISECONDS",
      "DURATION_IN",
      "DURATION_OUT",
      "MIN_TTL",
      "MAX_TTL",
      "LONGEST_FLOW_PKT",
      "SHORTEST_FLOW_PKT",
      "MIN_IP_PKT_LEN",
      "MAX_IP_PKT_LEN",
      "SRC_TO_DST_SECOND_BYTES",
      "DST_TO_SRC_SECOND_BYTES",
      "RETRANSMITTED_IN_BYTES",
      "RETRANSMITTED_IN_PKTS",
      "RETRANSMITTED_OUT_BYTES",
      "RETRANSMITTED_OUT_PKTS",
      "SRC_TO_DST_AVG_THROUGHPUT",
      "DST_TO_SRC_AVG_THROUGHPUT",
      "NUM_PKTS_UP_TO_128_BYTES",
      "NUM_PKTS_128_TO_256_BYTES",
      "NUM_PKTS_256_TO_512_BYTES",
      "NUM_PKTS_512_TO_1024_BYTES",
      "NUM_PKTS_1024_TO_1514_BYTES",
      "TCP_WIN_MAX_IN",
      "TCP_WIN_MAX_OUT",
      "ICMP_TYPE",
      "ICMP_IPV4_TYPE",
      "DNS_QUERY_ID",
      "DNS_QUERY_TYPE",
      "DNS_TTL_ANSWER",
      "FTP_COMMAND_RET_CODE"
    ],
    "portable_core_29": [
      "IN_BYTES",
      "OUT_BYTES",
      "IN_PKTS",
      "OUT_PKTS",
      "FLOW_DURATION_MILLISECONDS",
      "DURATION_IN",
      "DURATION_OUT",
      "MIN_TTL",
      "MAX_TTL",
      "LONGEST_FLOW_PKT",
      "SHORTEST_FLOW_PKT",
      "MIN_IP_PKT_LEN",
      "MAX_IP_PKT_LEN",
      "SRC_TO_DST_SECOND_BYTES",
      "DST_TO_SRC_SECOND_BYTES",
      "RETRANSMITTED_IN_BYTES",
      "RETRANSMITTED_IN_PKTS",
      "RETRANSMITTED_OUT_BYTES",
      "RETRANSMITTED_OUT_PKTS",
      "SRC_TO_DST_AVG_THROUGHPUT",
      "DST_TO_SRC_AVG_THROUGHPUT",
      "NUM_PKTS_UP_TO_128_BYTES",
      "NUM_PKTS_128_TO_256_BYTES",
      "NUM_PKTS_256_TO_512_BYTES",
      "NUM_PKTS_512_TO_1024_BYTES",
      "NUM_PKTS_1024_TO_1514_BYTES",
      "TCP_WIN_MAX_IN",
      "TCP_WIN_MAX_OUT",
      "DNS_TTL_ANSWER"
    ],
    "excluded_identifiers_3": [
      "IPV4_SRC_ADDR",
      "IPV4_DST_ADDR",
      "DNS_QUERY_ID"
    ],
    "semantic_codes_11": [
      "L4_SRC_PORT",
      "L4_DST_PORT",
      "PROTOCOL",
      "L7_PROTO",
      "TCP_FLAGS",
      "CLIENT_TCP_FLAGS",
      "SERVER_TCP_FLAGS",
      "ICMP_TYPE",
      "ICMP_IPV4_TYPE",
      "DNS_QUERY_TYPE",
      "FTP_COMMAND_RET_CODE"
    ]
  },
  "preprocessing_policy": {
    "fit_scope": "source_benign_only",
    "portable_core_29": "log1p_then_source_median_iqr_then_clip",
    "clip_range": [
      -8.0,
      8.0
    ],
    "missing_values": "source_median; halt_and_review_contract_if_any_core_feature_exceeds_0.5_percent_nonstructural_missing",
    "raw_numeric_codes_forbidden": true,
    "target_refit_forbidden_in_zero_shot": true
  },
  "mask_groups_portable_core_29": {
    "bidirectional_volume": [
      "IN_BYTES",
      "OUT_BYTES",
      "IN_PKTS",
      "OUT_PKTS"
    ],
    "duration": [
      "FLOW_DURATION_MILLISECONDS",
      "DURATION_IN",
      "DURATION_OUT"
    ],
    "ttl": [
      "MIN_TTL",
      "MAX_TTL"
    ],
    "packet_extrema": [
      "LONGEST_FLOW_PKT",
      "SHORTEST_FLOW_PKT",
      "MIN_IP_PKT_LEN",
      "MAX_IP_PKT_LEN"
    ],
    "rates": [
      "SRC_TO_DST_SECOND_BYTES",
      "DST_TO_SRC_SECOND_BYTES",
      "SRC_TO_DST_AVG_THROUGHPUT",
      "DST_TO_SRC_AVG_THROUGHPUT"
    ],
    "retransmissions": [
      "RETRANSMITTED_IN_BYTES",
      "RETRANSMITTED_IN_PKTS",
      "RETRANSMITTED_OUT_BYTES",
      "RETRANSMITTED_OUT_PKTS"
    ],
    "packet_size_histogram": [
      "NUM_PKTS_UP_TO_128_BYTES",
      "NUM_PKTS_128_TO_256_BYTES",
      "NUM_PKTS_256_TO_512_BYTES",
      "NUM_PKTS_512_TO_1024_BYTES",
      "NUM_PKTS_1024_TO_1514_BYTES"
    ],
    "tcp_window": [
      "TCP_WIN_MAX_IN",
      "TCP_WIN_MAX_OUT"
    ],
    "dns_answer": [
      "DNS_TTL_ANSWER"
    ]
  },
  "features": [
    {
      "position": 1,
      "name": "IPV4_SRC_ADDR",
      "description": "IPv4 source address",
      "group": "identifier",
      "model_action": "drop"
    },
    {
      "position": 2,
      "name": "L4_SRC_PORT",
      "description": "Layer-4 source port",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_bucket_encoding"
    },
    {
      "position": 3,
      "name": "IPV4_DST_ADDR",
      "description": "IPv4 destination address",
      "group": "identifier",
      "model_action": "drop"
    },
    {
      "position": 4,
      "name": "L4_DST_PORT",
      "description": "Layer-4 destination port",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_bucket_encoding"
    },
    {
      "position": 5,
      "name": "PROTOCOL",
      "description": "IP protocol identifier",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_one_hot"
    },
    {
      "position": 6,
      "name": "L7_PROTO",
      "description": "Layer-7 protocol numeric code",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_source_vocabulary"
    },
    {
      "position": 7,
      "name": "IN_BYTES",
      "description": "Incoming bytes",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 8,
      "name": "IN_PKTS",
      "description": "Incoming packets",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 9,
      "name": "OUT_BYTES",
      "description": "Outgoing bytes",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 10,
      "name": "OUT_PKTS",
      "description": "Outgoing packets",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 11,
      "name": "TCP_FLAGS",
      "description": "Cumulative TCP flags",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_bit_decomposition"
    },
    {
      "position": 12,
      "name": "CLIENT_TCP_FLAGS",
      "description": "Cumulative client TCP flags",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_bit_decomposition"
    },
    {
      "position": 13,
      "name": "SERVER_TCP_FLAGS",
      "description": "Cumulative server TCP flags",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_bit_decomposition"
    },
    {
      "position": 14,
      "name": "FLOW_DURATION_MILLISECONDS",
      "description": "Flow duration in milliseconds",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 15,
      "name": "DURATION_IN",
      "description": "Client-to-server stream duration",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 16,
      "name": "DURATION_OUT",
      "description": "Server-to-client stream duration",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 17,
      "name": "MIN_TTL",
      "description": "Minimum flow TTL",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 18,
      "name": "MAX_TTL",
      "description": "Maximum flow TTL",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 19,
      "name": "LONGEST_FLOW_PKT",
      "description": "Longest packet in the flow",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 20,
      "name": "SHORTEST_FLOW_PKT",
      "description": "Shortest packet in the flow",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 21,
      "name": "MIN_IP_PKT_LEN",
      "description": "Minimum IP packet length",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 22,
      "name": "MAX_IP_PKT_LEN",
      "description": "Maximum IP packet length",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 23,
      "name": "SRC_TO_DST_SECOND_BYTES",
      "description": "Source-to-destination bytes per second",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 24,
      "name": "DST_TO_SRC_SECOND_BYTES",
      "description": "Destination-to-source bytes per second",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 25,
      "name": "RETRANSMITTED_IN_BYTES",
      "description": "Retransmitted source-to-destination bytes",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 26,
      "name": "RETRANSMITTED_IN_PKTS",
      "description": "Retransmitted source-to-destination packets",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 27,
      "name": "RETRANSMITTED_OUT_BYTES",
      "description": "Retransmitted destination-to-source bytes",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 28,
      "name": "RETRANSMITTED_OUT_PKTS",
      "description": "Retransmitted destination-to-source packets",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 29,
      "name": "SRC_TO_DST_AVG_THROUGHPUT",
      "description": "Source-to-destination average throughput",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 30,
      "name": "DST_TO_SRC_AVG_THROUGHPUT",
      "description": "Destination-to-source average throughput",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 31,
      "name": "NUM_PKTS_UP_TO_128_BYTES",
      "description": "Packets with IP size up to 128 bytes",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 32,
      "name": "NUM_PKTS_128_TO_256_BYTES",
      "description": "Packets with IP size between 128 and 256 bytes",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 33,
      "name": "NUM_PKTS_256_TO_512_BYTES",
      "description": "Packets with IP size between 256 and 512 bytes",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 34,
      "name": "NUM_PKTS_512_TO_1024_BYTES",
      "description": "Packets with IP size between 512 and 1024 bytes",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 35,
      "name": "NUM_PKTS_1024_TO_1514_BYTES",
      "description": "Packets with IP size between 1024 and 1514 bytes",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 36,
      "name": "TCP_WIN_MAX_IN",
      "description": "Maximum source-to-destination TCP window",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 37,
      "name": "TCP_WIN_MAX_OUT",
      "description": "Maximum destination-to-source TCP window",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 38,
      "name": "ICMP_TYPE",
      "description": "ICMP type times 256 plus ICMP code",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_bucket_encoding"
    },
    {
      "position": 39,
      "name": "ICMP_IPV4_TYPE",
      "description": "ICMP type",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_bucket_encoding"
    },
    {
      "position": 40,
      "name": "DNS_QUERY_ID",
      "description": "DNS transaction identifier",
      "group": "identifier",
      "model_action": "drop"
    },
    {
      "position": 41,
      "name": "DNS_QUERY_TYPE",
      "description": "DNS query type",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_bucket_encoding"
    },
    {
      "position": 42,
      "name": "DNS_TTL_ANSWER",
      "description": "TTL of the first DNS A record",
      "group": "portable_numeric",
      "model_action": "portable_core_log_robust"
    },
    {
      "position": 43,
      "name": "FTP_COMMAND_RET_CODE",
      "description": "FTP client command return code",
      "group": "semantic_code",
      "model_action": "exclude_from_core_optional_return_class_encoding"
    }
  ]
}''', '82c4883eb190ce331aa5f338c2460238e533aad40be7bc1265c1601b978217f0')


### flow_pipeline/portable_general.py (incorporato)

In [ ]:
_pfs_put('flow_pipeline/portable_general.py', r'''"""Shared, fit-free PFS conversion. No training or dataset scan at import time."""
from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

CONTRACT_PATH = Path(__file__).resolve().parents[1] / 'feature_contracts' / 'portable_general29_v1.json'
CONTRACT = json.loads(CONTRACT_PATH.read_text(encoding='utf-8'))
FEATURES = tuple(item['name'] for item in CONTRACT['features'])
INDEX = {name: i for i, name in enumerate(FEATURES)}
if len(FEATURES) != 29 or CONTRACT['feature_count'] != len(FEATURES) or len(INDEX) != len(FEATURES):
    raise ValueError('Contratto PFS generale: numero o nomi delle feature non validi.')
UGR_COLUMNS = ('ts', 'td', 'sa', 'da', 'sp', 'dp', 'pr', 'flg', 'fwd', 'stos', 'ipkt', 'ibyt', 'label')
UGR_LABELS = ('background', 'dos', 'scan11', 'scan44', 'nerisbotnet', 'blacklist',
              'anomaly-udpscan', 'anomaly-sshscan', 'anomaly-spam')
NF_DOMAINS = ('NF-CSE-CIC-IDS2018-v2', 'NF-UNSW-NB15-v2', 'NF-ToN-IoT-v2', 'NF-BoT-IoT-v2')
FLAG_BITS = {'fin': 1, 'syn': 2, 'rst': 4, 'psh': 8, 'ack': 16, 'urg': 32, 'ece': 64, 'cwr': 128}
FLAG_CHARS = {'fin': 'F', 'syn': 'S', 'rst': 'R', 'psh': 'P', 'ack': 'A', 'urg': 'U', 'ece': 'E', 'cwr': 'C'}


def key(value):
    return ''.join(c for c in str(value).lower() if c.isalnum())


def profile_indices(name):
    names = CONTRACT['profiles'][name]
    return np.arange(len(FEATURES)) if names == 'all' else np.array([INDEX[n] for n in names])


def convert(frame, exporter, cic_byte_proxy=True):
    """One exported record -> 29 values and quality codes (0 missing, 1 proxy, 2 direct/derived).

    Missing, invalid and zero are separate. Rates at duration=0 stay missing.
    Aggregating forward/backward is explicit: this does not harmonize flow segmentation.
    """
    n = len(frame)
    values = np.full((n, len(FEATURES)), np.nan, np.float64)
    quality = np.zeros_like(values, np.uint8)
    capability = np.zeros(len(FEATURES), np.uint8)
    columns = {}
    for column in frame.columns:
        columns.setdefault(key(column), []).append(column)

    def take(*aliases):
        found = list(dict.fromkeys(c for a in aliases for c in columns.get(key(a), [])))
        if len(found) > 1:
            raise ValueError('Colonne ambigue per ' + str(aliases) + ': ' + str(found))
        return None if not found else frame[found[0]]

    def number(*aliases, factor=1., integer=False, maximum=None):
        column = take(*aliases)
        if column is None:
            return np.full(n, np.nan), False
        raw = pd.to_numeric(column, errors='coerce').to_numpy(dtype=float, na_value=np.nan) * factor
        valid = np.isfinite(raw) & (raw >= 0)
        if integer:
            valid &= raw == np.floor(raw)
        if maximum is not None:
            valid &= raw <= maximum
        return np.where(valid, raw, np.nan), True

    def put(name, raw, cap):
        j = INDEX[name]
        valid = np.isfinite(raw) & (raw >= 0)
        values[:, j] = np.where(valid, raw, np.nan)
        quality[:, j] = np.where(valid, cap, 0)
        capability[j] = cap

    def direct(name, aliases, factor=1., proxy=False, integer=False, maximum=None):
        raw, present = number(*aliases, factor=factor, integer=integer, maximum=maximum)
        put(name, raw, (1 if proxy else 2) if present else 0)
        return raw

    if exporter == 'netflow_v2':
        fwd_p, fp = number('IN_PKTS', integer=True)
        bwd_p, bp = number('OUT_PKTS', integer=True)
        fwd_b, fb = number('IN_BYTES', integer=True)
        bwd_b, bb = number('OUT_BYTES', integer=True)
        put('packets', fwd_p + bwd_p, 2 if fp and bp else 0)
        put('ip_bytes', fwd_b + bwd_b, 2 if fb and bb else 0)
        direct('duration_ms', ['FLOW_DURATION_MILLISECONDS'])
        direct('ip_packet_min_bytes', ['MIN_IP_PKT_LEN'])
        direct('ip_packet_max_bytes', ['MAX_IP_PKT_LEN'])
        direct('tcp_window_max_fwd', ['TCP_WIN_MAX_IN'])
        direct('tcp_window_max_bwd', ['TCP_WIN_MAX_OUT'])
        flags, flags_present = number('TCP_FLAGS', integer=True, maximum=255)
    elif exporter == 'cicflowmeter':
        fwd_p, fp = number('Total Fwd Packets', 'Tot Fwd Pkts', integer=True)
        bwd_p, bp = number('Total Backward Packets', 'Tot Bwd Pkts', integer=True)
        payload_f, pf = number('Total Length of Fwd Packets', 'TotLen Fwd Pkts')
        payload_b, pb = number('Total Length of Bwd Packets', 'TotLen Bwd Pkts')
        h_f, hf = number('Fwd Header Length', 'Fwd Header Len')
        h_b, hb = number('Bwd Header Length', 'Bwd Header Len')
        fwd_b = payload_f + h_f + 20. * fwd_p
        bwd_b = payload_b + h_b + 20. * bwd_p
        fb, bb = bool(cic_byte_proxy and pf and hf and fp), bool(cic_byte_proxy and pb and hb and bp)
        put('ip_bytes', fwd_b + bwd_b if fb and bb else np.full(n, np.nan), 1 if fb and bb else 0)
        put('packets', fwd_p + bwd_p, 2 if fp and bp else 0)
        direct('duration_ms', ['Flow Duration', 'Flow Duration usec'], factor=.001)
        direct('ip_packet_min_bytes', ['Min Packet Length', 'Pkt Len Min'], proxy=True)
        direct('ip_packet_max_bytes', ['Max Packet Length', 'Pkt Len Max'], proxy=True)
        direct('tcp_window_max_fwd', ['Init_Win_bytes_forward', 'Init Fwd Win Byts'], proxy=True)
        direct('tcp_window_max_bwd', ['Init_Win_bytes_backward', 'Init Bwd Win Byts'], proxy=True)
        for flag in FLAG_BITS:
            aliases = [flag.upper() + ' Flag Count', flag.upper() + ' Flag Cnt']
            # Historical CIC calls CWR 'CWE'; retains a declared proxy, not silently exact.
            cap = 2
            if flag == 'cwr' and take(*aliases) is None:
                aliases, cap = ['CWE Flag Count'], 1
            counts, present = number(*aliases, integer=True)
            put('tcp_' + flag, np.where(np.isfinite(counts), (counts > 0).astype(float), np.nan), cap if present else 0)
    elif exporter == 'ugr16':
        fwd_p, fp = number('ipkt', integer=True)
        fwd_b, fb = number('ibyt', integer=True)
        bwd_p = bwd_b = np.full(n, np.nan)
        bp = bb = False
        put('packets', fwd_p, 2 if fp else 0)
        put('ip_bytes', fwd_b, 2 if fb else 0)
        direct('duration_ms', ['td'], factor=1000.)
        flag_column = take('flg')
        strings = [] if flag_column is None else flag_column.fillna('').astype(str).str.strip().str.upper().tolist()
        for flag, letter in FLAG_CHARS.items():
            raw = np.full(n, np.nan)
            for row, text in enumerate(strings):
                # Six-flag nfdump export: absent ECE/CWR are unknown, not false.
                allowed = 'UAPRSF' if len(text) <= 6 else 'CEUAPRSF'
                if not text or len(text) > 8 or any(c not in allowed + '.' for c in text):
                    continue
                if flag in ('ece', 'cwr') and len(text) <= 6:
                    continue
                raw[row] = float(letter in text)
            cap = 2 if flag_column is not None and (flag not in ('ece', 'cwr') or np.isfinite(raw).any()) else 0
            put('tcp_' + flag, raw, cap)
    else:
        raise ValueError('Exporter non supportato: ' + exporter)

    if exporter == 'netflow_v2':
        valid = np.isfinite(flags)
        safe_flags = np.where(valid, flags, 0).astype(np.uint16)
        for flag, bit in FLAG_BITS.items():
            put('tcp_' + flag, np.where(valid, ((safe_flags & bit) != 0).astype(float), np.nan), 2 if flags_present else 0)

    protocol_column = take('PROTOCOL', 'Protocol', 'pr')
    protocol = np.full(n, np.nan)
    if protocol_column is not None:
        text = protocol_column.astype('string').fillna('').str.strip().str.upper()
        protocol = pd.to_numeric(text, errors='coerce').to_numpy(dtype=float, na_value=np.nan).copy()
        for name, code in {'TCP': 6, 'UDP': 17, 'ICMP': 1, 'ICMP6': 58, 'ICMPV6': 58}.items():
            protocol[text.eq(name).to_numpy(dtype=bool)] = code
        protocol[~(np.isfinite(protocol) & (protocol >= 0) & (protocol <= 255) & (protocol == np.floor(protocol)))] = np.nan
    for name, codes in [('tcp', (6,)), ('udp', (17,)), ('icmp', (1, 58))]:
        put('protocol_' + name, np.where(np.isfinite(protocol), np.isin(protocol, codes).astype(float), np.nan), 2 if protocol_column is not None else 0)
    # Ports are meaningful here only with an observed TCP/UDP protocol.
    for direction, aliases in [('src', ['L4_SRC_PORT', 'Src Port', 'Source Port', 'sp']),
                               ('dst', ['L4_DST_PORT', 'Dst Port', 'Destination Port', 'dp'])]:
        port, present = number(*aliases, integer=True, maximum=65535)
        valid = np.isfinite(port) & np.isin(protocol, [6, 17])
        cap = 2 if present and protocol_column is not None else 0
        for name, lo, hi in [('system', 0, 1023), ('user', 1024, 49151), ('dynamic', 49152, 65535)]:
            put(direction + '_port_' + name, np.where(valid, ((port >= lo) & (port <= hi)).astype(float), np.nan), cap)

    def ratio(name, numerator, denominator, factor=1.):
        a, b = INDEX[numerator], INDEX[denominator]
        valid = (quality[:, a] > 0) & (quality[:, b] > 0) & (values[:, b] > 0)
        raw = np.full(n, np.nan)
        np.divide(values[:, a] * factor, values[:, b], out=raw, where=valid)
        put(name, raw, min(capability[a], capability[b]))
        quality[:, INDEX[name]] = np.where(np.isfinite(raw), np.minimum(quality[:, a], quality[:, b]), 0)

    ratio('mean_ip_packet_bytes', 'ip_bytes', 'packets')
    ratio('byte_rate_s', 'ip_bytes', 'duration_ms', 1000.)
    ratio('packet_rate_s', 'packets', 'duration_ms', 1000.)
    if exporter != 'ugr16':
        for name, back, total, cap in [('reverse_packet_share', bwd_p, 'packets', 2 if bp and fp else 0),
                                      ('reverse_byte_share', bwd_b, 'ip_bytes', 1 if exporter == 'cicflowmeter' and fb and bb else 2 if fb and bb else 0)]:
            denominator = values[:, INDEX[total]]
            valid = np.isfinite(back) & (denominator > 0) & (quality[:, INDEX[total]] > 0)
            raw = np.full(n, np.nan)
            np.divide(back, denominator, out=raw, where=valid)
            put(name, raw, cap)
    # Do not present a missing structural field as an observed zero.
    values[quality == 0] = np.nan
    valid_record = (quality[:, INDEX['packets']] == 2) & (values[:, INDEX['packets']] > 0) & (quality[:, INDEX['duration_ms']] == 2)
    return {'values': values, 'quality': quality, 'capability': capability, 'valid_record': valid_record,
            'scope': CONTRACT['scope'], 'exporter': exporter}


def inventory(root):
    root = Path(root).resolve()
    result = {}
    for domain in NF_DOMAINS:
        paths = sorted((root / 'dataset_netflow' / domain).rglob(domain + '.csv'))
        if len(paths) != 1:
            raise ValueError('Atteso un solo CSV raw per ' + domain)
        result[domain] = {'exporter': 'netflow_v2', 'paths': paths, 'lineage': 'CSE2018' if 'CSE-CIC' in domain else domain}
    for domain, folder, lineage in [('CIC2017', 'dataset_final_2017', 'CIC2017'), ('CIC2018', 'dataset_final_2018', 'CSE2018')]:
        paths = sorted((root / folder).glob('*.csv'))
        if not paths:
            raise FileNotFoundError(root / folder)
        result[domain] = {'exporter': 'cicflowmeter', 'paths': paths, 'lineage': lineage}
    path = root / 'dataset_netflow_v9' / 'august.week1.csv'
    if not path.exists():
        raise FileNotFoundError(path)
    result['UGR16'] = {'exporter': 'ugr16', 'paths': [path], 'lineage': 'UGR16'}
    return result


def fingerprints(specs):
    return {d: [{'path': str(p.resolve()), 'bytes': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns}
                for p in s['paths']] for d, s in specs.items()}


def iter_frames(spec, rows_per_file, chunk=10000, counters=None):
    """Bounded pilot/preflight prefix per CSV; None scans the full collection.

    UGR budget counts physical CSV records including rejected/blank records.
    No on_bad_lines='skip': malformed quoted CSV interrupts, with no false completion.
    """
    counters = {} if counters is None else counters
    counters.update(records_read=0, rejected=0, blank=0, files=0)
    for path in spec['paths']:
        counters['files'] += 1
        if spec['exporter'] != 'ugr16':
            with pd.read_csv(path, dtype=str, chunksize=chunk, nrows=rows_per_file) as reader:
                for frame in reader:
                    counters['records_read'] += len(frame)
                    yield path, frame
            continue
        with path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.reader(handle, strict=True)
            batch, read = [], 0
            while rows_per_file is None or read < rows_per_file:
                try:
                    fields = next(reader)
                except StopIteration:
                    break
                read += 1
                counters['records_read'] += 1
                if not fields:
                    counters['blank'] += 1
                    continue
                if len(fields) != 13:
                    counters['rejected'] += 1
                    continue
                batch.append(fields)
                if len(batch) >= chunk:
                    yield path, pd.DataFrame(batch, columns=UGR_COLUMNS)
                    batch = []
            if batch:
                yield path, pd.DataFrame(batch, columns=UGR_COLUMNS)


def labels(frame, exporter):
    columns = {key(c): c for c in frame.columns}
    if 'label' not in columns:
        raise ValueError('Label assente: valutazione impossibile.')
    text = frame[columns['label']].astype('string').fillna('').str.strip()
    y = np.full(len(frame), -1, np.int8)
    if exporter == 'netflow_v2':
        numeric = pd.to_numeric(text, errors='coerce')
        valid = numeric.isin([0, 1]).to_numpy(dtype=bool)
        y[valid] = numeric[valid].to_numpy(dtype=np.int8)
        names = frame[columns['attack']].fillna('unknown').astype(str) if 'attack' in columns else text
    elif exporter == 'ugr16':
        names = text.str.lower()
        known = names.isin(UGR_LABELS).to_numpy(dtype=bool)
        y[known] = names[known].ne('background').to_numpy(dtype=np.int8)
    else:
        names = text
        known = ~text.str.lower().isin(['', 'nan', 'unknown', 'none', '<na>']).to_numpy(dtype=bool)
        y[known] = ~text[known].str.upper().isin(['BENIGN', 'NORMAL']).to_numpy(dtype=bool)
    return y, np.asarray(names, dtype=str)


def preflight(root, rows_per_file=1000, cic_byte_proxy=True):
    if type(rows_per_file) is not int or rows_per_file < 1:
        raise ValueError('Budget positivo richiesto.')
    specs = inventory(root)
    before = fingerprints(specs)
    rows, parsers, label_counts = [], {}, {}
    for domain, spec in specs.items():
        counters, per_file = {}, {}
        for path, frame in iter_frames(spec, rows_per_file, counters=counters):
            converted = convert(frame, spec['exporter'], cic_byte_proxy)
            y, _ = labels(frame, spec['exporter'])
            class_counts = label_counts.setdefault(domain, {}).setdefault(Path(path).name,
                {'benign': 0, 'attack': 0, 'unknown': 0})
            for name, code in [('benign', 0), ('attack', 1), ('unknown', -1)]:
                class_counts[name] += int((y == code).sum())
            q, v = converted['quality'], converted['values']
            record = per_file.setdefault(str(path), {'rows': 0, 'invalid_records': 0, 'observed': np.zeros(29, int),
                'proxy': np.zeros(29, int), 'zeros': np.zeros(29, int), 'capability': converted['capability']})
            record['rows'] += len(frame)
            record['invalid_records'] += int((~converted['valid_record']).sum())
            record['observed'] += (q > 0).sum(axis=0)
            record['proxy'] += (q == 1).sum(axis=0)
            record['zeros'] += ((q > 0) & (v == 0)).sum(axis=0)
            profiles_to_check = ('common9', 'common12_bytes_proxy') if cic_byte_proxy else ('common9',)
            for profile in profiles_to_check:
                ids = profile_indices(profile)
                if np.any(converted['capability'][ids] == 0):
                    missing = [FEATURES[j] for j in ids if converted['capability'][j] == 0]
                    raise ValueError(domain + ': profilo ' + profile + ' non disponibile: ' + str(missing))
        parsers[domain] = counters
        if not per_file:
            raise ValueError('Nessun record nel budget: ' + domain)
        for path, data in per_file.items():
            for j, feature in enumerate(FEATURES):
                count = int(data['observed'][j])
                rows.append({'domain': domain, 'file': Path(path).name, 'feature': feature,
                    'rows': data['rows'], 'invalid_records': data['invalid_records'],
                    'capability': int(data['capability'][j]), 'observed_rows': count,
                    'proxy_rows': int(data['proxy'][j]), 'missing_rows': data['rows'] - count,
                    'zero_fraction_observed': float(data['zeros'][j] / count) if count else None,
                    'primary_common9': j in profile_indices('common9')})
    if fingerprints(specs) != before:
        raise ValueError('CSV modificati durante il preflight.')
    return {'contract_version': CONTRACT['version'], 'contract_sha256': hashlib.sha256(CONTRACT_PATH.read_bytes()).hexdigest(),
        'scope': 'prefix_per_file_diagnostic', 'rows_per_file': rows_per_file,
        'cic_byte_proxy': cic_byte_proxy, 'input_fingerprints': before, 'features': rows, 'parsers': parsers,
        'label_counts_in_prefixes': label_counts,
        'training_started': False, 'dataset_rebuilt': False,
        'notice': 'Header e prefissi non certificano semantica dell’estrattore, intero dataset o importanza predittiva.'}


def fold_plan(specs):
    """Actual seven-domain LODO, excluding every known representation of the target capture."""
    return [{'target': target, 'sources': [d for d in specs if specs[d]['lineage'] != spec['lineage']],
             'excluded_same_capture': [d for d in specs if d != target and specs[d]['lineage'] == spec['lineage']]}
            for target, spec in specs.items()]
''', '05f96bdb754f9bd95c01e20336ef32dfc4710ac492cbf104a4deebd3084e3a39')


### flow_pipeline/portable_general_semantics.py (incorporato)

In [ ]:
_pfs_put('flow_pipeline/portable_general_semantics.py', r'''"""Conservative eligibility for the known local CSV collection, separate from mapping.

No repaired values, exporter-version inference, training or scans on import.
Historical converter outputs remain intact for comparison and provenance.
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

from flow_pipeline import portable_general as pfs

POLICY_PATH = Path(__file__).resolve().parents[1] / 'feature_contracts' / 'portable_general_semantics_v1.json'
POLICY = json.loads(POLICY_PATH.read_text(encoding='utf-8'))
if POLICY['catalog'] != pfs.CONTRACT['version']:
    raise ValueError('Politica semantica e catalogo PFS non corrispondono.')
for rule in POLICY['rules'] + POLICY['row_rules']:
    if set(rule['features']) - set(pfs.FEATURES) or set(rule['domains']) - set(POLICY['domains']):
        raise ValueError('Regola semantica con feature o domini sconosciuti: ' + rule['id'])


def reject_retired_profiles(profiles):
    retired = sorted(set(profiles) & set(POLICY['retired_training_profiles']))
    if retired:
        raise ValueError('Profili storici sospesi dopo la revisione semantica dei flag CIC: ' +
                         ', '.join(retired) + '. Eseguire solo Dataset, sezione 16; '
                         'i report precedenti restano consultabili. Questo audit non abilita un nuovo fit.')


def apply_semantic_policy(converted, domain, view='original'):
    """Return an independent eligible view; never change mapped values/quality.

    A rule applies to a known collection, not every future CSV from an exporter.
    Missing and excluded values share quality=0, with separate reasons/counts.
    """
    if domain not in POLICY['domains'] or POLICY['domains'][domain] != converted['exporter']:
        raise ValueError('Dominio/exporter non coperto dalla politica semantica: ' + str(domain))
    if view not in ('original', 'final'):
        raise ValueError('Vista semantica non supportata: ' + str(view))
    values = np.array(converted['values'], dtype=np.float64, copy=True)
    quality = np.array(converted['quality'], dtype=np.uint8, copy=True)
    capability = np.array(converted['capability'], dtype=np.uint8, copy=True)
    original_quality = np.asarray(converted['quality'])
    reasons = {name: [] for name in pfs.FEATURES}
    rules_applied = []
    for rule in POLICY['rules']:
        if domain not in rule['domains']:
            continue
        rules_applied.append(rule['id'])
        for name in rule['features']:
            j = pfs.INDEX[name]
            quality[:, j] = 0
            capability[j] = 0
            reasons[name].append(rule['id'])
    if converted['exporter'] == 'netflow_v2':
        j = pfs.INDEX['ip_packet_min_bytes']
        packets = values[:, pfs.INDEX['packets']]
        excluded = (original_quality[:, j] > 0) & (values[:, j] == 0) & (packets > 0)
        if excluded.any():
            quality[excluded, j] = 0
            reasons['ip_packet_min_bytes'].append('nf_zero_ip_minimum')
            rules_applied.append('nf_zero_ip_minimum')
    # Propagate exclusions/proxy quality to exact derivations. Fixed point also
    # handles future dependency chains, without treating zero as missing.
    for _ in range(len(pfs.FEATURES)):
        previous = quality.copy(), capability.copy()
        previous_reasons = {name: tuple(codes) for name, codes in reasons.items()}
        for child, parents in POLICY['dependencies'].items():
            j = pfs.INDEX[child]
            for parent in parents:
                k = pfs.INDEX[parent]
                quality[:, j] = np.minimum(quality[:, j], quality[:, k])
                capability[j] = min(capability[j], capability[k])
                if reasons[parent]:
                    reasons[child] = list(dict.fromkeys(reasons[child] + reasons[parent]))
        if (np.array_equal(previous[0], quality) and np.array_equal(previous[1], capability)
                and previous_reasons == {name: tuple(codes) for name, codes in reasons.items()}):
            break
    values[quality == 0] = np.nan
    features = []
    for j, name in enumerate(pfs.FEATURES):
        mapped_cap = int(converted['capability'][j])
        cap = int(capability[j])
        features.append({
            'feature': name, 'mapping_capability': mapped_cap, 'eligible_capability': cap,
            'status': 'missing' if mapped_cap == 0 else 'excluded' if cap == 0 else
                      'proxy_candidate' if cap == 1 else 'exact_mapping_candidate',
            'excluded_observed_rows': int(((original_quality[:, j] > 0) & (quality[:, j] == 0)).sum()),
            'eligible_exact_rows': int((quality[:, j] == 2).sum()),
            'eligible_proxy_rows': int((quality[:, j] == 1).sum()),
            'unavailable_rows': int((quality[:, j] == 0).sum()),
            'exclusion_reasons': reasons[name],
        })
    return {'values': values, 'quality': quality, 'capability': capability,
            'features': features, 'rules_applied': rules_applied, 'policy_version': POLICY['version'],
            'notice': POLICY['notice']}


def eligibility_tables(file_reports):
    matrix, domains = [], []
    for report in file_reports:
        matrix.extend({'domain': report['domain'], 'view': report['view'], 'file': report['file'], **f}
                      for f in report['eligibility']['features'])
    for domain, view in dict.fromkeys((r['domain'], r['view']) for r in file_reports):
        group = [r for r in file_reports if r['domain'] == domain and r['view'] == view]
        mapped = np.min([[f['mapping_capability'] for f in r['eligibility']['features']] for r in group], axis=0)
        caps = np.min([[f['eligible_capability'] for f in r['eligibility']['features']] for r in group], axis=0)
        domains.append({
            'domain': domain, 'view': view, 'files': len(group),
            'exact_mapping_candidates_on_every_file': int((caps == 2).sum()),
            'proxy_candidates_on_every_file': int((caps == 1).sum()),
            'semantically_excluded_on_at_least_one_file': int(((mapped > 0) & (caps == 0)).sum()),
            'structurally_missing_on_at_least_one_file': int((mapped == 0).sum()),
            'excluded_observed_cells': sum(f['excluded_observed_rows'] for r in group for f in r['eligibility']['features']),
            'notice': 'Conteggi del catalogo dopo esclusioni; non tutte le righe sono valide/osservate e non sono informazioni indipendenti.',
        })
    originals = [r for r in file_reports if r['view'] == 'original']
    caps = np.min([[f['eligible_capability'] for f in r['eligibility']['features']] for r in originals], axis=0)
    observed = np.min([[f['eligible_exact_rows'] + f['eligible_proxy_rows']
                        for f in r['eligibility']['features']] for r in originals], axis=0)
    exact = [pfs.FEATURES[j] for j in np.flatnonzero((caps == 2) & (observed > 0))]
    with_proxy = [pfs.FEATURES[j] for j in np.flatnonzero((caps > 0) & (observed > 0))]
    exploratory = [n for n in POLICY['exploratory_common_volume'] if n in with_proxy]
    return matrix, domains, {
        'exact_mapping_candidates_on_every_original_file': exact,
        'candidates_allowing_declared_proxies_on_every_original_file': with_proxy,
        'exploratory_common_volume_coordinates': exploratory,
        'exploratory_volume_complete_in_samples': len(exploratory) == len(POLICY['exploratory_common_volume']),
        'exploratory_volume_notice': POLICY['exploratory_common_volume_notice'],
        'all29_exact_eligible_on_every_original_file': len(exact) == 29,
        'accepted_training_profile': None, 'ready_for_training': False,
        'next_decision': 'Valutare un PFS a nucleo ridotto con estensioni dichiarate, oppure CSV CIC riestratti con tool verificato. Nessun riempimento artificiale fino a 29.',
        'notice': POLICY['notice'],
    }
''', 'aa6e831ad390394f6236cce68d8af48dc28e17815097a9e95513964b49d5ff69')


### models/portable_general_experiment.py (incorporato)

In [ ]:
_pfs_put('models/portable_general_experiment.py', r'''"""Fit-free helpers for the current PFS experiment; obsolete training is disabled."""
from __future__ import annotations
import hashlib
import json
import os
from decimal import Decimal, InvalidOperation
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
from flow_pipeline.portable_general import CONTRACT, FEATURES, inventory, key
from flow_pipeline.portable_general_semantics import reject_retired_profiles
ROLE_BOUNDARIES = np.array([.80, .84, .86, .88, .90])
ROLE_NAMES = {0: 'train', 1: 'stop', 3: 'calibration'}
SPLIT_NOTICE = 'Grouped native-record hash split; no proof of session/host/capture disjointness. Historically inspected data; exploratory.'
def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)

def native_roles(frame, exporter, nf_roles):
    if exporter == 'netflow_v2':
        result = np.asarray(nf_roles(frame), np.uint8)
        if result.shape != (len(frame),) or np.any(result > 5):
            raise ValueError('Ruoli V7 non validi.')
        return result

    @lru_cache(maxsize=65536)
    def canonical(text):
        text = str(text).strip().lower()
        if text in ('', 'nan', 'none', 'null', '<na>'):
            return '?'
        try:
            number = Decimal(text)
        except InvalidOperation:
            return text
        if not number.is_finite():
            return text
        if number == 0:
            return '0'
        sign, digits, exponent = number.as_tuple()
        digits = list(digits)
        while digits[-1] == 0:
            digits.pop()
            exponent += 1
        return ('-' if sign else '') + ''.join(map(str, digits)) + 'e' + str(exponent)

    columns = sorted((key(c), c) for c in frame.columns if key(c) not in {'label', 'attack', 'dataset', 'class'})
    if len({k for k, _ in columns}) != len(columns):
        raise ValueError('Colonne native duplicate: gruppi ambigui.')
    arrays = []
    for _, column in columns:
        strings = frame[column].fillna('').astype(str)
        mapping = {text: canonical(text) for text in strings.unique()}
        arrays.append(strings.map(mapping).to_numpy())
    prefix = json.dumps([exporter, [k for k, _ in columns], 'pfs_general_roles_v1_seed42']).encode()
    draws = np.array([int.from_bytes(hashlib.sha256(prefix + json.dumps(list(row), separators=(',', ':')).encode()).digest()[:4], 'big')
                      for row in zip(*arrays)], np.uint64)
    return np.searchsorted(np.floor(ROLE_BOUNDARIES * 2**32).astype(np.uint64), draws, side='right').astype(np.uint8)

def raw_budget(domain, cfg):
    return cfg.get('raw_rows_by_domain', {}).get(domain, cfg['raw_rows_per_domain'])

def feature_weights(ids):
    groups = [CONTRACT['features'][j]['group'] for j in ids]
    distinct = set(groups)
    return np.asarray([1. / (len(distinct) * groups.count(g)) for g in groups], np.float32)

def lineage_weights(sources, specs):
    groups = {specs[d]['lineage'] for d in sources}
    return {d: 1. / (len(groups) * sum(specs[s]['lineage'] == specs[d]['lineage'] for s in sources)) for d in sources}

def fit_scaler(pools, sources, specs, ids, allow_proxy):
    medians, scales = np.zeros(len(ids)), np.ones(len(ids))
    mixing = lineage_weights(sources, specs)
    observed_counts = {}
    for k, j in enumerate(ids):
        item = CONTRACT['features'][j]
        arrays, masses = [], []
        for domain in sources:
            pool = pools[domain]['train']
            valid = pool['quality'][:, j] >= (1 if allow_proxy else 2)
            raw = pool['values'][valid, j].astype(float)
            observed_counts[domain + '|' + item['name']] = len(raw)
            if len(raw):
                arrays.append(np.log1p(raw) if item['kind'] == 'continuous' else raw)
                masses.append(np.full(len(raw), mixing[domain] / len(raw)))
        if not arrays:
            raise ValueError('Feature senza osservazioni sorgente TRAIN: ' + item['name'])
        if item['kind'] != 'continuous':
            continue
        raw, weights = np.concatenate(arrays), np.concatenate(masses)
        order = np.argsort(raw, kind='stable')
        raw, weights = raw[order], weights[order]
        cumulative = np.cumsum(weights) / weights.sum()
        q10, q25, q50, q75, q90 = np.interp([.10, .25, .50, .75, .90], cumulative, raw)
        medians[k], scales[k] = q50, max(q75-q25, (q90-q10)/2.563, .25)
    return {'feature_names': [FEATURES[j] for j in ids], 'indices': ids.tolist(),
        'median': medians.tolist(), 'scale': scales.tolist(), 'clip': 8., 'allow_proxy': allow_proxy,
        'source_weights': mixing, 'observed_train_rows': observed_counts, 'fit_scope': 'source_benign_train_only'}

def transform(pool, scaler):
    ids = np.asarray(scaler['indices'])
    values = pool['values'][:, ids].astype(float)
    observed = pool['quality'][:, ids] >= (1 if scaler['allow_proxy'] else 2)
    # Impute only after preserving the observed mask; missing terms never enter the loss/score.
    for k, j in enumerate(ids):
        if CONTRACT['features'][j]['kind'] == 'continuous':
            values[:, k] = np.log1p(values[:, k])
    scaled = (values - scaler['median']) / scaler['scale']
    clipped = np.clip(np.where(observed, scaled, 0.), -scaler['clip'], scaler['clip']).astype(np.float32)
    if not np.isfinite(clipped).all():
        raise ValueError('Input normalizzato non finito.')
    return clipped, observed.astype(np.float32)

def metrics(tn, fp, fn, tp):
    precision, recall = tp/max(tp+fp, 1), tp/max(tp+fn, 1)
    return {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp, 'precision': precision, 'recall': recall,
            'f1': 2*tp/max(2*tp+fp+fn, 1), 'specificity': tn/max(tn+fp, 1),
            'accuracy': (tn+tp)/max(tn+fp+fn+tp, 1)}

def run_pilot(root, output_root, cfg, nf_roles, mode='pilot'):
    """Compatibility guard: retired protocols cannot scan inputs or fit models."""
    if mode not in ('sources', 'pilot'):
        raise ValueError('Modalita eseguibili: sources o pilot.')
    reject_retired_profiles(cfg['profiles'])
    raise ValueError('Usare esclusivamente il pilot PFS mascherato corrente.')''', '4db478e41ffb902e530cfa0e5f32f2833a397958d92da21e6db6895df067271a')


### flow_pipeline/portable_general_audit.py (incorporato)

In [ ]:
_pfs_put('flow_pipeline/portable_general_audit.py', r'''"""Bounded PFS information audit, explicitly run by the user from Dataset.

Reads original CIC columns and final CSVs independently. No positional joins,
training, feature selection on labels, full scans, or source rewrites.
"""
from __future__ import annotations

import csv
import hashlib
import json
import os
import uuid
from collections import Counter
from html import escape
from pathlib import Path

import numpy as np
import pandas as pd

from flow_pipeline import portable_general as pfs
from flow_pipeline import portable_general_semantics as semantics

AUDIT_VERSION = 'portable_general_information_audit_v3_semantic_eligibility'
OFFSETS = (0., .25, .50, .75)
TIME_FEATURES = ('duration_ms', 'byte_rate_s', 'packet_rate_s')
META_ALIASES = {
    'source_port': ('Src Port', 'Source Port'),
    'destination_port': ('Dst Port', 'Destination Port'),
    'protocol': ('Protocol',), 'timestamp': ('Timestamp',),
    'source_address': ('Src IP', 'Source IP'),
    'destination_address': ('Dst IP', 'Destination IP'),
    'flow_id': ('Flow ID',), 'cwr_historical_alias': ('CWE Flag Count',),
}
PROXY_NOTES = {
    'ip_bytes': 'CIC: stima IPv4 senza opzioni, payload + header trasporto + 20*pacchetti.',
    'mean_ip_packet_bytes': 'Eredita la qualità dei byte; su CIC dipende dalla stima IP.',
    'byte_rate_s': 'Eredita la qualità dei byte; indefinito quando durata=0.',
    'reverse_byte_share': 'Su CIC dipende dalla stessa stima dei byte IP.',
    'ip_packet_min_bytes': 'CIC: minimo di payload usato come proxy del minimo IP.',
    'ip_packet_max_bytes': 'CIC: massimo di payload usato come proxy del massimo IP.',
    'tcp_window_max_fwd': 'CIC: finestra iniziale usata come proxy del massimo forward.',
    'tcp_window_max_bwd': 'CIC: finestra iniziale usata come proxy del massimo backward.',
    'tcp_cwr': 'CIC: alias storico CWE conservato come proxy dichiarato.',
}


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def file_identity(path):
    path = Path(path).resolve()
    stat = path.stat()
    return {'path': str(path), 'bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns}


def audit_output_paths(root, identity, nonce):
    """Keep output paths short even in the project's deeply nested Windows checkout."""
    if len(identity) != 16 or len(nonce) != 8 or any(c not in '0123456789abcdef' for c in identity + nonce):
        raise ValueError('Identità/nonce di output non validi.')
    base = Path(root).resolve() / 'outputs' / 'pfsa'
    final = base / identity
    stage = base / ('.p' + nonce + '.partial')
    longest = max(len(str(parent / 's' / '99' / 'original_columns.csv')) for parent in (final, stage))
    if os.name == 'nt' and longest > 240:
        raise ValueError('Percorso output Windows ancora troppo lungo (' + str(longest) +
                         ' caratteri): usare un percorso più corto per il progetto.')
    return base, final, stage


def sample_relative_directory(index):
    """File/domain/view remain in report.json, not repeated in directory names."""
    if type(index) is not int or not 0 <= index <= 99:
        raise ValueError('Indice campione fuori dal layout corto 00..99.')
    return Path('s') / f'{index:02d}'


def audit_inventory(root):
    """Originals are the recovered input view; old final data remain comparators."""
    root = Path(root).resolve()
    specs = pfs.inventory(root)
    records = []
    for domain, spec in specs.items():
        if spec['exporter'] != 'cicflowmeter':
            records.extend({'domain': domain, 'view': 'original', 'exporter': spec['exporter'],
                            'path': path} for path in spec['paths'])
            continue
        folder = root / ('dataset' if domain == 'CIC2017' else 'dataset_2018')
        originals = {path.name: path for path in folder.glob('*.csv')}
        final_names = {path.name for path in spec['paths']}
        if set(originals) != final_names:
            raise ValueError('Inventario CIC original/final differente per ' + domain + ': ' +
                             str(sorted(set(originals) ^ final_names)))
        for path in spec['paths']:
            records.append({'domain': domain, 'view': 'original', 'exporter': 'cicflowmeter',
                            'path': originals[path.name]})
            records.append({'domain': domain, 'view': 'final', 'exporter': 'cicflowmeter', 'path': path})
    return records


def unique_columns(header):
    """Preserve duplicate-column values under explicit, reproducible suffixes."""
    columns, seen, duplicates = [], set(), []
    for index, original in enumerate(header):
        name = str(original).strip()
        if not name or name.startswith('_pfs_audit_'):
            raise ValueError('Nome colonna vuoto/riservato nell’header.')
        proposed, suffix = name, 1
        while proposed in seen:
            proposed = name + '__duplicate_' + str(suffix)
            suffix += 1
        if proposed != name:
            duplicates.append({'original': original, 'column': proposed, 'position': index,
                               'compared_with': name})
        columns.append(proposed)
        seen.add(proposed)
    if len({pfs.key(c) for c in columns}) != len(columns):
        raise ValueError('Alias normalizzati ambigui nell’header: ' + str(columns))
    return columns, duplicates


def sample_windows(path, exporter, rows_per_window=1000, fractions=OFFSETS,
                   max_bytes_per_window=8 * 1024**2, max_line_bytes=64 * 1024):
    """Seek byte windows in one-physical-line CSVs, validating every parsed row.

    Fractions are byte positions, not row/time quantiles or random samples.
    Blank/malformed/repeated-header lines consume the same rigid line budget.
    Multiline quoted records and invalid UTF-8 interrupt rather than fabricate rows.
    """
    if type(rows_per_window) is not int or rows_per_window < 1:
        raise ValueError('rows_per_window deve essere un intero positivo.')
    fractions = tuple(fractions)
    if not fractions or any(not np.isfinite(f) or not 0 <= f < 1 for f in fractions):
        raise ValueError('Frazioni byte richieste in [0,1).')
    if tuple(sorted(set(fractions))) != fractions:
        raise ValueError('Frazioni byte uniche e ordinate richieste.')
    if max_line_bytes < 1 or max_bytes_per_window < max_line_bytes:
        raise ValueError('Budget byte/linea incoerenti.')
    path = Path(path)
    before = file_identity(path)
    records, offsets, window_ids, windows, seen_offsets = [], [], [], [], set()
    with path.open('rb') as handle:
        if exporter == 'ugr16':
            header, data_start = list(pfs.UGR_COLUMNS), 0
        else:
            raw_header = handle.readline(max_line_bytes + 1)
            if len(raw_header) > max_line_bytes or not raw_header:
                raise ValueError('Header assente/troppo lungo: ' + str(path))
            header = next(csv.reader([raw_header.decode('utf-8-sig')], strict=True))
            data_start = handle.tell()
        columns, duplicates = unique_columns(header)
        for window_index, fraction in enumerate(fractions):
            desired = max(data_start, int(before['bytes'] * fraction))
            handle.seek(desired)
            alignment_bytes = 0
            if desired > data_start:
                handle.seek(desired - 1)
                previous = handle.read(1)
                if previous != b'\n':
                    partial = handle.readline(max_line_bytes + 1)
                    alignment_bytes = len(partial)
                    if alignment_bytes > max_line_bytes:
                        raise ValueError('Linea troppo lunga durante allineamento: ' + str(path))
            start = handle.tell()
            info = {'fraction_of_file_bytes': fraction, 'start_byte': start,
                    'lines_read': 0, 'accepted': 0, 'blank': 0, 'wrong_field_count': 0,
                    'repeated_header': 0, 'duplicate_offsets': 0,
                    'bytes_read': alignment_bytes, 'ended_at_eof': False, 'reject_examples': []}
            for _ in range(rows_per_window):
                if info['bytes_read'] >= max_bytes_per_window:
                    break
                offset = handle.tell()
                raw = handle.readline(min(max_line_bytes, max_bytes_per_window - info['bytes_read']) + 1)
                if not raw:
                    info['ended_at_eof'] = True
                    break
                if len(raw) > max_line_bytes or info['bytes_read'] + len(raw) > max_bytes_per_window:
                    raise ValueError('Budget byte/linea superato: ' + str(path))
                info['lines_read'] += 1
                info['bytes_read'] += len(raw)
                if offset in seen_offsets:
                    info['duplicate_offsets'] += 1
                    continue
                seen_offsets.add(offset)
                if not raw.strip():
                    info['blank'] += 1
                    continue
                try:
                    row = next(csv.reader([raw.decode('utf-8-sig')], strict=True))
                except (UnicodeDecodeError, csv.Error) as error:
                    raise ValueError('CSV non UTF-8 o record su più linee, offset ' + str(offset) +
                                     ' in ' + str(path) + ': ' + str(error)) from error
                if len(row) != len(header):
                    info['wrong_field_count'] += 1
                    if len(info['reject_examples']) < 3:
                        info['reject_examples'].append({'byte_offset': offset, 'fields': len(row)})
                    continue
                if [value.strip() for value in row] == [value.strip() for value in header]:
                    info['repeated_header'] += 1
                    continue
                records.append(row)
                offsets.append(offset)
                window_ids.append(window_index)
                info['accepted'] += 1
            info['byte_budget_reached'] = info['bytes_read'] >= max_bytes_per_window
            windows.append(info)
    if before != file_identity(path):
        raise ValueError('CSV modificato durante il campionamento: ' + str(path))
    frame = pd.DataFrame(records, columns=columns, dtype=str)
    if not len(frame):
        raise ValueError('Nessun record accettato nelle finestre: ' + str(path))
    provenance = pd.DataFrame({'_pfs_audit_source_byte': offsets, '_pfs_audit_window': window_ids})
    return frame, provenance, {'fingerprint': before, 'original_header': header,
                              'columns': columns, 'duplicate_columns': duplicates, 'windows': windows}


def numeric(frame, *aliases):
    selected = [c for c in frame.columns if pfs.key(c) in {pfs.key(a) for a in aliases}]
    if len(selected) > 1:
        raise ValueError('Colonne ambigue nell’audit: ' + str(selected))
    if not selected:
        return np.full(len(frame), np.nan)
    return pd.to_numeric(frame[selected[0]], errors='coerce').to_numpy(dtype=float, na_value=np.nan)


def metadata_fields(frame):
    result = []
    for name, aliases in META_ALIASES.items():
        columns = [c for c in frame.columns if pfs.key(c) in {pfs.key(a) for a in aliases}]
        result.append({'field': name, 'columns': columns, 'present': bool(columns),
                       'nonempty_rows': int(frame[columns[0]].str.strip().ne('').sum()) if columns else 0})
    return result


def feature_statistics(converted):
    result = []
    for j, definition in enumerate(pfs.CONTRACT['features']):
        quality, values = converted['quality'][:, j], converted['values'][:, j]
        observed = values[quality > 0]
        quantiles = np.quantile(observed, [0, .01, .5, .99, 1]).tolist() if len(observed) else [None] * 5
        result.append({'feature': pfs.FEATURES[j], 'unit': definition['unit'],
            'definition': definition['definition'], 'capability': int(converted['capability'][j]),
            'rows': len(values), 'exact_rows': int((quality == 2).sum()),
            'proxy_rows': int((quality == 1).sum()), 'missing_rows': int((quality == 0).sum()),
            'zero_observed_rows': int((observed == 0).sum()),
            'constant_observed': bool(len(observed) and observed.min() == observed.max()),
            **dict(zip(('min', 'p01', 'median', 'p99', 'max'), quantiles)),
            'proxy_note': PROXY_NOTES.get(pfs.FEATURES[j], '')})
    return result


def semantic_diagnostics(frame, converted, exporter, provenance, duplicates=()):
    """Comparisons are review signals, not a repair or proof of an exporter bug."""
    checks = []
    values = converted['values']
    packets, duration = values[:, pfs.INDEX['packets']], values[:, pfs.INDEX['duration_ms']]

    def add(code, eligible, condition, features, note, examples_fields=()):
        eligible = np.asarray(eligible, bool)
        hit = eligible & np.asarray(condition, bool)
        examples = []
        for row in np.flatnonzero(hit)[:3]:
            example = {'source_byte': int(provenance.iloc[row]['_pfs_audit_source_byte'])}
            for name, array in examples_fields:
                value = array[row]
                example[name] = float(value) if np.isfinite(value) else None
            examples.append(example)
        checks.append({'check': code, 'compared_rows': int(eligible.sum()),
                       'flagged_rows': int(hit.sum()), 'features_to_review': list(features),
                       'note': note, 'examples': examples})

    available = np.isfinite(duration) & np.isfinite(packets) & (packets > 0)
    add('zero_duration', available, duration == 0, TIME_FEATURES,
        'Durata zero conservata; rate mancanti. Il campione non ne stabilisce la causa.')
    if exporter == 'netflow_v2':
        fwd, bwd = numeric(frame, 'DURATION_IN'), numeric(frame, 'DURATION_OUT')
        span = np.maximum(fwd, bwd)
        both = available & np.isfinite(span) & (fwd >= 0) & (bwd >= 0)
        examples = [('duration_ms', duration), ('directional_max_ms', span)]
        add('nf_total_shorter_than_directional_span', both, duration + 1 < span, TIME_FEATURES,
            'Confronto tra campi dichiarati in ms; nessuna sostituzione con durata direzionale.', examples)
        add('nf_near_unsigned32_microseconds_pattern', both & (span > 0),
            (duration > 4_000_000) & np.isclose(duration + span, 2**32 / 1000., rtol=0, atol=2),
            TIME_FEATURES, 'Pattern numerico sospetto, non prova di overflow né formula di correzione.', examples)
        total, client, server = [numeric(frame, name) for name in ('TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS')]
        valid = np.isfinite(total) & np.isfinite(client) & np.isfinite(server)
        valid &= (total >= 0) & (client >= 0) & (server >= 0) & (total <= 255) & (client <= 255) & (server <= 255)
        valid &= (total == np.floor(total)) & (client == np.floor(client)) & (server == np.floor(server))
        union = np.bitwise_or(np.where(valid, client, 0).astype(np.uint16),
                              np.where(valid, server, 0).astype(np.uint16))
        add('nf_total_flags_vs_directional_union', valid, total != union,
            ['tcp_' + flag for flag in pfs.FLAG_BITS],
            'Confronto della maschera totale con OR delle due direzioni; differenze da chiarire.')
        minimum = values[:, pfs.INDEX['ip_packet_min_bytes']]
        add('nf_zero_ip_minimum_with_packets', np.isfinite(minimum) & np.isfinite(packets) & (packets > 0),
            minimum == 0, ['ip_packet_min_bytes'],
            'Possibile sentinella: raw e qualità del mapping conservati, correttezza fisica da chiarire.')
    if exporter == 'cicflowmeter':
        raw_duration = numeric(frame, 'Flow Duration')
        add('cic_negative_or_nonfinite_duration', np.ones(len(frame), bool),
            ~np.isfinite(raw_duration) | (raw_duration < 0), TIME_FEATURES,
            'Durata raw invalida: il converter conserva il valore mancante, senza imputare.')
        observed_rate = numeric(frame, 'Flow Packets/s', 'Flow Pkts/s', 'Flow Packets per sec')
        expected = np.full(len(frame), np.nan)
        np.divide(packets * 1000., duration, out=expected, where=available & (duration > 0))
        compared = available & (duration > 0) & np.isfinite(observed_rate)
        add('cic_reported_packet_rate_vs_counters', compared,
            ~np.isclose(observed_rate, expected, rtol=1e-4, atol=1e-3), TIME_FEATURES,
            'Confronto del rate CIC con pacchetti/durata convertiti; tolleranza esplicita.',
            [('reported_packets_per_s', observed_rate), ('derived_packets_per_s', expected)])
        for flag in pfs.FLAG_BITS:
            aliases = (flag.upper() + ' Flag Count', flag.upper() + ' Flag Cnt')
            if flag == 'cwr':
                aliases += ('CWE Flag Count',)
            count = numeric(frame, *aliases)
            add('cic_' + flag + '_count_exceeds_packets', np.isfinite(count) & np.isfinite(packets),
                (count > packets) | (count < 0) | (count != np.floor(count)), ['tcp_' + flag],
                'Conteggio non intero/negativo o maggiore dei pacchetti: semantica da verificare.')
            finite = count[np.isfinite(count)]
            binary_only = bool(len(finite) and np.isin(finite, [0, 1]).all())
            add('cic_' + flag + '_binary_only_count_in_multipacket_sample',
                np.isfinite(count) & np.isfinite(packets) & (packets > 1),
                np.full(len(frame), binary_only), ['tcp_' + flag],
                'Tutti i conteggi osservati sono 0/1 anche con flussi multipacchetto. '
                'Compatibile con il difetto storico del primo pacchetto, ma non ne prova la causa. '
                'Non si ricostruiscono presenze cumulative dai conteggi direzionali.',
                [('flag_count', count), ('packets', packets)])
        for flag in ('psh', 'urg'):
            fwd = numeric(frame, 'Fwd ' + flag.upper() + ' Flags')
            bwd = numeric(frame, 'Bwd ' + flag.upper() + ' Flags')
            total = numeric(frame, flag.upper() + ' Flag Count', flag.upper() + ' Flag Cnt')
            compared = np.isfinite(fwd) & np.isfinite(bwd) & np.isfinite(total)
            add('cic_' + flag + '_directional_presence_vs_total', compared,
                (np.maximum(fwd, bwd) > 0) != (total > 0), ['tcp_' + flag],
                'Presenze direzionali e totale discordanti; non si assume che il CSV storico implementi lo stesso conteggio.')
    for duplicate in duplicates:
        left, right = frame[duplicate['compared_with']].str.strip(), frame[duplicate['column']].str.strip()
        a, b = pd.to_numeric(left, errors='coerce'), pd.to_numeric(right, errors='coerce')
        same = left.eq(right).to_numpy() | np.isclose(a.to_numpy(dtype=float), b.to_numpy(dtype=float),
                                                   rtol=0, atol=0, equal_nan=False)
        add('duplicate_column_' + duplicate['compared_with'], np.ones(len(frame), bool), ~same,
            [name for name in PROXY_NOTES if name != 'tcp_cwr'],
            'Colonne duplicate conservate; l’eventuale discordanza richiede revisione del proxy/header.')
    return checks


def coverage_tables(file_reports):
    feature_rows, domain_rows = [], []
    for report in file_reports:
        for feature in report['features']:
            feature_rows.append({'domain': report['domain'], 'view': report['view'],
                                 'file': report['file'], **feature})
    for domain, view in dict.fromkeys((r['domain'], r['view']) for r in file_reports):
        group = [r for r in file_reports if r['domain'] == domain and r['view'] == view]
        caps = np.min([[f['capability'] for f in r['features']] for r in group], axis=0)
        domain_rows.append({'domain': domain, 'view': view, 'files': len(group),
            'sample_rows': sum(r['rows'] for r in group),
            'exact_catalog_on_every_file': int((caps == 2).sum()),
            'proxy_catalog_on_every_file': int((caps == 1).sum()),
            'not_available_on_every_file': int((caps == 0).sum()),
            'invalid_records': sum(r['invalid_records'] for r in group),
            'benign_sample_rows': sum(r['label_counts']['benign'] for r in group),
            'attack_sample_rows': sum(r['label_counts']['attack'] for r in group),
            'unknown_label_sample_rows': sum(r['label_counts']['unknown'] for r in group),
            'rejected_candidate_lines': sum(w['blank'] + w['wrong_field_count'] + w['repeated_header']
                                            for r in group for w in r['parser']['windows']),
            'semantic_checks_flagged': sum(c['flagged_rows'] > 0 for r in group for c in r['semantic_checks'])})
    original = [r for r in file_reports if r['view'] == 'original']
    caps = np.min([[f['capability'] for f in r['features']] for r in original], axis=0)
    common = [pfs.FEATURES[j] for j in np.flatnonzero(caps == 2)]
    suspect = set()
    for report in original:
        suspect.update(f['feature'] for f in report.get('eligibility', {}).get('features', [])
                       if f['exclusion_reasons'])
        for check in report['semantic_checks']:
            if check['flagged_rows'] and check['check'] != 'zero_duration':
                suspect.update(check['features_to_review'])
        suspect.update(f['feature'] for f in report['features'] if f['exact_rows'] == 0)
    return feature_rows, domain_rows, {
        'structurally_exact_on_every_original_file': common,
        'exact_common_feature_count': len(common),
        'all29_exact_on_every_original_file': len(common) == 29,
        'common_features_with_review_signals': [name for name in common if name in suspect],
        'common_features_without_detected_signals_in_samples': [name for name in common if name not in suspect],
        'accepted_training_profile': None,
        'notice': 'Candidati, non selezione definitiva: mapping esatto non certifica valori, timeout o importanza.'}


def recovery_table(file_reports):
    result = []
    finals = {(r['domain'], r['file']): r for r in file_reports if r['view'] == 'final'}
    for original in (r for r in file_reports if r['view'] == 'original' and r['exporter'] == 'cicflowmeter'):
        final = finals[(original['domain'], original['file'])]
        final_meta = {f['field']: f for f in final['metadata']}
        for field in original['metadata']:
            result.append({'domain': original['domain'], 'file': original['file'], 'field': field['field'],
                'original_present': field['present'], 'final_present': final_meta[field['field']]['present'],
                'recovered_in_sample': bool(field['present'] and not final_meta[field['field']]['present']),
                'original_nonempty_sample_rows': field['nonempty_rows'],
                'columns': ', '.join(field['columns']),
                'note': 'Timestamp/IP/flow ID sono provenienza, non input AE. Porta senza protocollo non viene inferita.'})
    return result


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')


def render_report(report):
    def table(rows):
        if not rows:
            return '<p>Nessuna riga.</p>'
        return pd.DataFrame(rows).to_html(index=False, escape=True, border=0)
    anomalies = [{'domain': r['domain'], 'view': r['view'], 'file': r['file'],
                  'check': c['check'], 'compared_rows': c['compared_rows'],
                  'flagged_rows': c['flagged_rows'], 'note': c['note']}
                 for r in report['files'] for c in r['semantic_checks'] if c['flagged_rows']]
    recovered = [r for r in report['recovery'] if r['recovered_in_sample']]
    return ('<!doctype html><html lang="it"><meta charset="utf-8"><title>Audit informazioni PFS</title>'
            '<style>body{font:15px system-ui;margin:28px}table{border-collapse:collapse;margin:18px 0}'
            'th,td{padding:7px;border:1px solid #ccc;text-align:left}th{background:#eee}</style>'
            '<h1>Audit informazioni PFS: catalogo 29</h1><p>' + escape(report['notice']) + '</p>'
            '<h2>Copertura minima fra i file di ciascun dominio</h2>' + table(report['domain_summary']) +
            '<h2>Utilizzabilità dopo la revisione semantica</h2>' + table(report['eligibility_summary']) +
            '<p>La qualità del mapping è conservata. Il CSV eligible29 esclude valori da rivedere, '
            'senza ripararli o sostituirli. Le ragioni sono nella matrice di utilizzabilità.</p>'
            '<h2>Nucleo ed estensioni possibili</h2><pre>' +
            escape(json.dumps(report['semantic_candidates'], indent=2, ensure_ascii=False)) + '</pre>' +
            '<h2>Colonne originali recuperate nei campioni</h2>' + table(recovered) +
            '<h2>Relazioni da chiarire</h2>' + table(anomalies) +
            '<h2>Candidati comuni, soggetti a revisione</h2><pre>' +
            escape(json.dumps(report['core_candidates'], indent=2, ensure_ascii=False)) + '</pre>'
            '<p>I CSV campionari conservano tutte le colonne e le etichette delle righe originali accettate. '
            'Non sono dataset completi per il training. Nessuna equivalenza di sessioni, direzioni o timeout è certificata.</p></html>')


def verify_saved_audit(directory, seal):
    directory = Path(directory)
    manifest = json.loads((directory / 'manifest.json').read_text(encoding='utf-8'))
    if manifest['seal'] != seal or manifest['status'] != 'complete':
        raise ValueError('Audit salvato non coerente con configurazione/codice/input.')
    for name, digest in manifest['artifacts_sha256'].items():
        if sha256_file(directory / name) != digest:
            raise ValueError('Artefatto audit modificato: ' + name)
    return json.loads((directory / 'report.json').read_text(encoding='utf-8'))


def run_information_audit(root, rows_per_window=1000, fractions=OFFSETS,
                          cic_byte_proxy=True, fixtures=None, progress=print):
    root = Path(root).resolve()
    records = audit_inventory(root)
    before = [file_identity(r['path']) for r in records]
    code_paths = [Path(__file__).resolve().parents[1] / 'notebook_runtime.py', Path(__file__), Path(pfs.__file__), pfs.CONTRACT_PATH,
                  Path(semantics.__file__), semantics.POLICY_PATH,
                  Path(__file__).resolve().parents[1] / 'models' / 'portable_general_experiment.py',
                  root / 'tools' / 'test_portable_general.py', root / 'tools' / 'test_portable_general_audit.py']
    seal = {'version': AUDIT_VERSION, 'rows_per_window': rows_per_window, 'fractions': list(fractions),
            'max_bytes_per_window': 8 * 1024**2, 'max_line_bytes': 64 * 1024,
            'cic_byte_proxy': bool(cic_byte_proxy), 'inputs': before,
            'runtime': {'numpy': np.__version__, 'pandas': pd.__version__},
            'code_sha256': {str(path.resolve()): sha256_file(path) for path in code_paths}, 'fixtures': fixtures}
    identity = hashlib.sha256(json.dumps(seal, sort_keys=True, allow_nan=False).encode()).hexdigest()[:16]
    base, final, stage = audit_output_paths(root, identity, uuid.uuid4().hex[:8])
    base.mkdir(parents=True, exist_ok=True)
    if final.exists():
        report = verify_saved_audit(final, seal)
        if [file_identity(r['path']) for r in records] != before:
            raise ValueError('Input cambiati durante verifica audit salvato.')
        progress('Audit già completo, verificati gli hash: ' + str(final))
    else:
        stage.mkdir(exist_ok=False)
        files = []
        for index, record in enumerate(records):
            progress('Audit ' + str(index + 1) + '/' + str(len(records)) + ': ' + record['domain'] +
                     ' / ' + record['view'] + ' / ' + record['path'].name)
            frame, provenance, parser = sample_windows(record['path'], record['exporter'], rows_per_window, fractions)
            converted = pfs.convert(frame, record['exporter'], cic_byte_proxy)
            binary, names = pfs.labels(frame, record['exporter'])
            relative = sample_relative_directory(index)
            sample_dir = stage / relative
            sample_dir.mkdir(parents=True, exist_ok=False)
            pd.concat([provenance, frame], axis=1).to_csv(sample_dir / 'original_columns.csv', index=False)
            derived = pd.DataFrame(converted['values'], columns=pfs.FEATURES)
            quality = pd.DataFrame(converted['quality'], columns=['quality_' + name for name in pfs.FEATURES])
            status = pd.DataFrame({'valid_record': converted['valid_record'], 'binary_label': binary,
                                   'attack_label': names})
            pd.concat([provenance, status, derived, quality], axis=1).to_csv(sample_dir / 'pfs29.csv', index=False)
            eligible = semantics.apply_semantic_policy(converted, record['domain'], record['view'])
            eligible_values = pd.DataFrame(eligible['values'], columns=pfs.FEATURES)
            eligible_quality = pd.DataFrame(eligible['quality'], columns=['eligible_quality_' + name for name in pfs.FEATURES])
            pd.concat([provenance, eligible_values, eligible_quality], axis=1).to_csv(
                sample_dir / 'eligible29.csv', index=False)
            features = feature_statistics(converted)
            checks = semantic_diagnostics(frame, converted, record['exporter'], provenance, parser['duplicate_columns'])
            files.append({'domain': record['domain'], 'view': record['view'], 'exporter': record['exporter'],
                'file': record['path'].name, 'sample_directory': str(relative), 'rows': len(frame),
                'invalid_records': int((~converted['valid_record']).sum()), 'parser': parser,
                'label_counts': {'benign': int((binary == 0).sum()), 'attack': int((binary == 1).sum()),
                                 'unknown': int((binary == -1).sum())},
                'attack_names': dict(Counter(names.tolist())), 'metadata': metadata_fields(frame),
                'features': features, 'semantic_checks': checks,
                'eligibility': {k: v for k, v in eligible.items() if k not in ('values', 'quality', 'capability')}})
        if before != [file_identity(r['path']) for r in records]:
            raise ValueError('Input modificati durante l’audit; risultato parziale non pubblicato.')
        features, summary, core = coverage_tables(files)
        eligibility_matrix, eligibility_summary, semantic_candidates = semantics.eligibility_tables(files)
        report = {'version': AUDIT_VERSION, 'artifact_id': identity, 'directory': str(final), 'seal': seal,
            'scope': 'bounded_byte_windows_per_file', 'files': files, 'feature_matrix': features,
            'domain_summary': summary, 'recovery': recovery_table(files), 'core_candidates': core,
            'semantic_policy': semantics.POLICY, 'eligibility_matrix': eligibility_matrix,
            'eligibility_summary': eligibility_summary, 'semantic_candidates': semantic_candidates,
            'training_started': False, 'scaler_fitted': False, 'sources_rewritten': False,
            'ready_for_training': False, 'full_datasets_prepared': False,
            'max_candidate_lines': len(records) * len(fractions) * rows_per_window,
            'notice': 'Audit campionario di informazioni e valori; nessun training. Finestre per byte '
                      'distribuite nel file, non campione casuale o certificazione del dataset completo.'}
        write_json(stage / 'report.json', report)
        pd.DataFrame(features).to_csv(stage / 'feature_matrix.csv', index=False)
        pd.DataFrame(summary).to_csv(stage / 'domain_summary.csv', index=False)
        pd.DataFrame(eligibility_matrix).to_csv(stage / 'eligibility_matrix.csv', index=False)
        pd.DataFrame(eligibility_summary).to_csv(stage / 'eligibility_summary.csv', index=False)
        pd.DataFrame(report['recovery']).to_csv(stage / 'recovered_metadata.csv', index=False)
        (stage / 'report.html').write_text(render_report(report), encoding='utf-8')
        artifacts = {str(path.relative_to(stage)): sha256_file(path) for path in sorted(stage.rglob('*')) if path.is_file()}
        write_json(stage / 'manifest.json', {'status': 'complete', 'seal': seal, 'artifacts_sha256': artifacts,
                                           'purpose': 'audit_samples_only_not_training_dataset'})
        stage.rename(final)
    pointer_payload = {'report': str(final / 'report.json'), 'sha256': sha256_file(final / 'report.json')}
    # Keep the previous pointer location usable without putting sample files there.
    for pointer in (base / 'latest.json', root / 'outputs' / 'pfs_general_audit_v2' / 'latest.json'):
        pointer.parent.mkdir(parents=True, exist_ok=True)
        temporary = pointer.parent / ('.l' + uuid.uuid4().hex[:8] + '.tmp')
        write_json(temporary, pointer_payload)
        os.replace(temporary, pointer)
    return report
''', '87e60df5fa4791c78b1297b2d5ebb41f263810411902b0ecfe4f82ca6c7e7ccc')


### tools/test_portable_general.py (incorporato)

In [ ]:
_pfs_put('tools/test_portable_general.py', r'''"""Semantic fixtures for the user to run; imports never execute tests automatically."""
import unittest

import numpy as np
import pandas as pd

from flow_pipeline.portable_general import FEATURES, INDEX, convert, fold_plan, labels, profile_indices


class PortableGeneralTests(unittest.TestCase):
    def fixtures(self):
        nf = pd.DataFrame({'IN_BYTES': [190], 'OUT_BYTES': [0], 'IN_PKTS': [3], 'OUT_PKTS': [0],
            'FLOW_DURATION_MILLISECONDS': [52], 'TCP_FLAGS': [26], 'PROTOCOL': [6],
            'L4_SRC_PORT': [52000], 'L4_DST_PORT': [443]})
        cic = pd.DataFrame({'Total Length of Fwd Packets': [70], 'Total Length of Bwd Packets': [0],
            'Total Fwd Packets': [3], 'Total Backward Packets': [0], 'Flow Duration': [52000],
            'Fwd Header Length': [60], 'Bwd Header Length': [0], 'FIN Flag Count': [0],
            'SYN Flag Count': [1], 'RST Flag Count': [0], 'PSH Flag Count': [1],
            'ACK Flag Count': [2], 'URG Flag Count': [0], 'ECE Flag Count': [0]})
        ugr = pd.DataFrame({'ibyt': [190], 'ipkt': [3], 'td': [.052], 'flg': ['.AP.S.'],
            'pr': ['TCP'], 'sp': [52000], 'dp': [443]})
        return nf, cic, ugr

    def test_common_profiles_compare_the_same_fixture_across_exporters(self):
        nf, cic, ugr = [convert(f, e) for f, e in zip(self.fixtures(), ['netflow_v2', 'cicflowmeter', 'ugr16'])]
        for profile in ('common9', 'common12_bytes_proxy'):
            ids = profile_indices(profile)
            np.testing.assert_allclose(nf['values'][:, ids], cic['values'][:, ids])
            np.testing.assert_allclose(nf['values'][:, ids], ugr['values'][:, ids])
        self.assertEqual(cic['quality'][0, INDEX['ip_bytes']], 1)
        self.assertEqual(cic['quality'][0, INDEX['mean_ip_packet_bytes']], 1)
        self.assertEqual(cic['values'][0, INDEX['tcp_ack']], 1)  # presence, not ACK packet count

    def test_zero_duration_is_observed_but_undefined_rate_is_missing(self):
        nf = self.fixtures()[0]
        nf['FLOW_DURATION_MILLISECONDS'] = 0
        result = convert(nf, 'netflow_v2')
        self.assertEqual(result['values'][0, INDEX['duration_ms']], 0)
        self.assertTrue(result['valid_record'][0])
        for name in ('byte_rate_s', 'packet_rate_s'):
            self.assertEqual(result['quality'][0, INDEX[name]], 0)
            self.assertTrue(np.isnan(result['values'][0, INDEX[name]]))

    def test_unavailable_reverse_direction_and_flags_are_not_zeros(self):
        result = convert(self.fixtures()[2], 'ugr16')
        for name in ('reverse_packet_share', 'reverse_byte_share', 'tcp_ece', 'tcp_cwr'):
            self.assertEqual(result['quality'][0, INDEX[name]], 0)
            self.assertTrue(np.isnan(result['values'][0, INDEX[name]]))
        self.assertEqual(result['quality'][0, INDEX['tcp_fin']], 2)
        self.assertEqual(result['values'][0, INDEX['tcp_fin']], 0)

    def test_no_protocol_or_ports_are_inferred(self):
        cic = self.fixtures()[1]
        cic['Destination Port'] = 443
        result = convert(cic, 'cicflowmeter')
        for name in ('protocol_tcp', 'protocol_udp', 'dst_port_system'):
            self.assertEqual(result['quality'][0, INDEX[name]], 0)

    def test_labels_never_affect_features(self):
        nf = self.fixtures()[0]
        left = convert(nf.assign(Label=0, Attack='Benign'), 'netflow_v2')
        right = convert(nf.assign(Label=1, Attack='DoS'), 'netflow_v2')
        np.testing.assert_allclose(left['values'], right['values'], equal_nan=True)
        np.testing.assert_array_equal(left['quality'], right['quality'])

    def test_labels_preserve_benign_attack_unknown_across_csv_types(self):
        cases = [('netflow_v2', ['0', '1', '?']),
                 ('cicflowmeter', [' BENIGN ', 'DDoS', 'unknown']),
                 ('ugr16', ['background', 'dos', 'unlisted'])]
        for exporter, values in cases:
            y, _ = labels(pd.DataFrame({'Label': pd.Series(values, dtype='string')}), exporter)
            np.testing.assert_array_equal(y, [0, 1, -1])

    def test_invalid_packet_counters_are_not_rounded_or_imputed(self):
        nf = self.fixtures()[0]
        for invalid in (-1, .5, np.inf):
            nf['IN_PKTS'] = invalid
            result = convert(nf, 'netflow_v2')
            self.assertFalse(result['valid_record'][0])
            self.assertEqual(result['quality'][0, INDEX['packets']], 0)

    def test_proxy_can_be_disabled(self):
        result = convert(self.fixtures()[1], 'cicflowmeter', cic_byte_proxy=False)
        for name in ('ip_bytes', 'byte_rate_s', 'mean_ip_packet_bytes', 'reverse_byte_share'):
            self.assertEqual(result['quality'][0, INDEX[name]], 0)

    def test_lodo_excludes_every_known_representation_of_target_capture(self):
        specs = {name: {'lineage': lineage} for name, lineage in [
            ('NF-CSE-CIC-IDS2018-v2', 'CSE2018'), ('CIC2018', 'CSE2018'),
            ('CIC2017', 'CIC2017'), ('UGR16', 'UGR16')]}
        for fold in fold_plan(specs):
            self.assertNotIn(fold['target'], fold['sources'])
            for domain in fold['sources']:
                self.assertNotEqual(specs[domain]['lineage'], specs[fold['target']]['lineage'])
        self.assertEqual(len(FEATURES), 29)

    def test_scaler_ignores_target_pool_and_transform_never_refits(self):
        import copy
        from models.portable_general_experiment import fit_scaler, transform

        source = convert(self.fixtures()[0], 'netflow_v2')
        target_frame = self.fixtures()[0]
        target_frame['IN_PKTS'] = 4096
        target_frame['FLOW_DURATION_MILLISECONDS'] = 52000
        target = convert(target_frame, 'netflow_v2')
        pools = {'source': {'train': source}, 'target': {'train': target}}
        specs = {'source': {'lineage': 'source'}, 'target': {'lineage': 'target'}}
        scaler = fit_scaler(pools, ['source'], specs, profile_indices('common9'), False)
        without_target = fit_scaler({'source': pools['source']}, ['source'], specs, profile_indices('common9'), False)
        self.assertEqual(scaler, without_target)
        frozen = copy.deepcopy(scaler)
        values, observed = transform(target, scaler)
        self.assertEqual(scaler, frozen)
        self.assertTrue(np.isfinite(values).all())
        self.assertEqual(values[0, scaler['feature_names'].index('packets')], 8)
        self.assertEqual(values[0, scaler['feature_names'].index('tcp_ack')], 1)
        self.assertEqual(observed.shape, values.shape)

    def test_missing_terms_are_imputed_only_with_an_explicit_zero_mask(self):
        from models.portable_general_experiment import fit_scaler, transform

        source = convert(self.fixtures()[0], 'netflow_v2')
        scaler = fit_scaler({'source': {'train': source}}, ['source'],
                            {'source': {'lineage': 'source'}}, profile_indices('common9'), False)
        missing = convert(self.fixtures()[0], 'netflow_v2')
        missing['values'][0, INDEX['packet_rate_s']] = np.nan
        missing['quality'][0, INDEX['packet_rate_s']] = 0
        values, observed = transform(missing, scaler)
        j = scaler['feature_names'].index('packet_rate_s')
        self.assertEqual(values[0, j], 0)
        self.assertEqual(observed[0, j], 0)
        self.assertEqual(observed[0, scaler['feature_names'].index('tcp_fin')], 1)


def run_fixtures():
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(PortableGeneralTests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise AssertionError('Fixture PFS generale fallite; non avviare training.')
    return {'tests_run': result.testsRun, 'passed': True}


if __name__ == '__main__':
    run_fixtures()
''', '74b2458c5dbc318ff84b83d08ee994e2eaddc8af99244c4033f6497a7cc512ab')


### tools/test_portable_general_audit.py (incorporato)

In [ ]:
_pfs_put('tools/test_portable_general_audit.py', r'''"""Information-audit regression fixtures. Run explicitly by the user, never on import."""
import json
import tempfile
import unittest
from pathlib import Path
from unittest.mock import patch

import numpy as np
import pandas as pd

from flow_pipeline import portable_general as pfs
from flow_pipeline import portable_general_semantics as semantics
from flow_pipeline.portable_general_audit import (feature_statistics, metadata_fields,
    sample_windows, semantic_diagnostics, unique_columns, verify_saved_audit, sha256_file,
    audit_output_paths, sample_relative_directory, run_information_audit)


class PortableGeneralAuditTests(unittest.TestCase):
    def test_output_creation_in_a_172_character_project_path(self):
        with tempfile.TemporaryDirectory() as directory:
            parent = Path(directory).resolve()
            if len(str(parent)) >= 171:
                self.skipTest('Il percorso temporaneo è già più lungo della fixture Windows.')
            root = parent / ('r' * (172 - len(str(parent)) - 1))
            root.mkdir()
            self.assertEqual(len(str(root)), 172)
            _, final, stage = audit_output_paths(root, '0123456789abcdef', '12345678')
            relative = sample_relative_directory(40)
            sample = stage / relative
            sample.mkdir(parents=True)
            csv_path = sample / 'original_columns.csv'
            csv_path.write_text('Label\nBENIGN\n', encoding='utf-8')
            self.assertLessEqual(len(str(csv_path)), 240)
            self.assertLessEqual(len(str(final / relative / csv_path.name)), 240)
            stage.rename(final)
            self.assertEqual((final / relative / csv_path.name).read_text(encoding='utf-8'), 'Label\nBENIGN\n')

    def test_sample_path_is_independent_of_long_source_names(self):
        self.assertEqual(sample_relative_directory(0), Path('s') / '00')
        self.assertEqual(sample_relative_directory(40), Path('s') / '40')
        with self.assertRaises(ValueError):
            sample_relative_directory(100)
        with self.assertRaises(ValueError):
            audit_output_paths(Path('.'), '../invalid', '12345678')

    def sample(self, content, **kwargs):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / 'fixture.csv'
            path.write_bytes(content)
            return sample_windows(path, 'cicflowmeter', **kwargs)

    def test_line_budget_accounts_for_blank_malformed_and_repeated_header(self):
        frame, provenance, parser = self.sample(
            b'a,b,Label\n1,2,BENIGN\ninvalid,row\n\na,b,Label\n3,4,DoS\n',
            rows_per_window=5, fractions=(0.,))
        window = parser['windows'][0]
        self.assertEqual(window['lines_read'], 5)
        self.assertEqual(window['accepted'], 2)
        self.assertEqual(window['blank'], 1)
        self.assertEqual(window['wrong_field_count'], 1)
        self.assertEqual(window['repeated_header'], 1)
        self.assertEqual(frame['Label'].tolist(), ['BENIGN', 'DoS'])
        self.assertEqual(frame['a'].tolist(), ['1', '3'])
        self.assertEqual(len(provenance), len(frame))

    def test_distributed_windows_preserve_alignment_and_never_duplicate_offsets(self):
        content = b'a,b,Label\n' + b''.join(
            ('%d,%d,BENIGN\n' % (i, i + 100)).encode() for i in range(300))
        frame, provenance, parser = self.sample(content, rows_per_window=3)
        self.assertEqual(len(frame), 12)
        self.assertEqual(provenance['_pfs_audit_source_byte'].nunique(), 12)
        self.assertEqual(len(parser['windows']), 4)
        self.assertTrue((frame['b'].astype(int) == frame['a'].astype(int) + 100).all())
        self.assertGreater(int(frame['a'].iloc[-1]), 150)

    def test_overlapping_windows_are_deduplicated(self):
        frame, provenance, parser = self.sample(b'a,b,Label\n1,2,BENIGN\n3,4,DoS\n',
                                              rows_per_window=10, fractions=(0., .01))
        self.assertEqual(len(frame), 2)
        self.assertEqual(provenance['_pfs_audit_source_byte'].nunique(), 2)
        self.assertEqual(parser['windows'][1]['duplicate_offsets'], 2)

    def test_multiline_csv_is_rejected_instead_of_inventing_records(self):
        with self.assertRaises(ValueError):
            self.sample(b'a,b,Label\n"quoted\nvalue",2,BENIGN\n', fractions=(0.,))

    def test_headerless_ugr_preserves_13_fields_and_accounts_for_rejects(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / 'ugr.csv'
            path.write_bytes(b'\nmalformed,row\n2016-08-01 00:10:09,0.052,1.2.3.4,5.6.7.8,56097,443,TCP,.AP.S.,0,0,3,190,background\n')
            frame, provenance, parser = sample_windows(path, 'ugr16', rows_per_window=3, fractions=(0.,))
            self.assertEqual(tuple(frame.columns), pfs.UGR_COLUMNS)
            self.assertEqual(parser['windows'][0]['blank'], 1)
            self.assertEqual(parser['windows'][0]['wrong_field_count'], 1)
            self.assertEqual(parser['windows'][0]['accepted'], 1)
            converted = pfs.convert(frame, 'ugr16')
            self.assertEqual(converted['values'][0, pfs.INDEX['ip_bytes']], 190)
            self.assertEqual(converted['quality'][0, pfs.INDEX['reverse_byte_share']], 0)
            self.assertEqual(frame['label'].iloc[0], 'background')
            self.assertEqual(len(provenance), 1)

    def test_duplicate_columns_and_their_values_are_preserved(self):
        frame, _, parser = self.sample(b'a,a,Label\n1,2,BENIGN\n', fractions=(0.,))
        self.assertEqual(frame['a'].tolist(), ['1'])
        self.assertEqual(frame['a__duplicate_1'].tolist(), ['2'])
        self.assertEqual(parser['duplicate_columns'][0]['position'], 1)
        with self.assertRaises(ValueError):
            unique_columns(['Dst Port', 'DstPort'])

    def test_recovered_port_is_not_inferred_to_have_a_protocol(self):
        frame = pd.DataFrame({'Destination Port': ['443'], 'Total Fwd Packets': ['2'],
                              'Total Backward Packets': ['0'], 'Flow Duration': ['1000000'],
                              'Label': ['BENIGN']})
        converted = pfs.convert(frame, 'cicflowmeter')
        self.assertEqual(converted['capability'][pfs.INDEX['dst_port_system']], 0)
        self.assertTrue(next(f for f in metadata_fields(frame) if f['field'] == 'destination_port')['present'])
        self.assertFalse(next(f for f in metadata_fields(frame) if f['field'] == 'protocol')['present'])
        self.assertEqual(frame['Label'].iloc[0], 'BENIGN')

    def test_suspicious_nf_duration_is_reported_without_repair(self):
        frame = pd.DataFrame({'IN_PKTS': [2], 'OUT_PKTS': [0], 'FLOW_DURATION_MILLISECONDS': [4294467],
                              'DURATION_IN': [500], 'DURATION_OUT': [0], 'TCP_FLAGS': [2]})
        converted = pfs.convert(frame, 'netflow_v2')
        provenance = pd.DataFrame({'_pfs_audit_source_byte': [123]})
        checks = semantic_diagnostics(frame, converted, 'netflow_v2', provenance)
        signal = next(c for c in checks if c['check'] == 'nf_near_unsigned32_microseconds_pattern')
        self.assertEqual(signal['flagged_rows'], 1)
        self.assertEqual(signal['examples'][0]['source_byte'], 123)
        self.assertEqual(converted['values'][0, pfs.INDEX['duration_ms']], 4294467)
        self.assertEqual(frame['FLOW_DURATION_MILLISECONDS'].iloc[0], 4294467)

    def test_directional_flags_and_total_are_compared_without_overwriting(self):
        frame = pd.DataFrame({'Total Fwd Packets': [2], 'Total Backward Packets': [0],
                              'Flow Duration': [1000], 'PSH Flag Count': [0],
                              'Fwd PSH Flags': [1], 'Bwd PSH Flags': [0]})
        converted = pfs.convert(frame, 'cicflowmeter')
        checks = semantic_diagnostics(frame, converted, 'cicflowmeter',
                                      pd.DataFrame({'_pfs_audit_source_byte': [100]}))
        signal = next(c for c in checks if c['check'] == 'cic_psh_directional_presence_vs_total')
        self.assertEqual(signal['flagged_rows'], 1)
        self.assertEqual(converted['values'][0, pfs.INDEX['tcp_psh']], 0)

    def test_statistics_distinguish_exact_zero_proxy_and_missing(self):
        frame = pd.DataFrame({'Total Fwd Packets': [2], 'Total Backward Packets': [0],
            'Flow Duration': [0], 'Total Length of Fwd Packets': [20],
            'Total Length of Bwd Packets': [0], 'Fwd Header Length': [40], 'Bwd Header Length': [0]})
        statistics = {f['feature']: f for f in feature_statistics(pfs.convert(frame, 'cicflowmeter'))}
        self.assertEqual(statistics['duration_ms']['exact_rows'], 1)
        self.assertEqual(statistics['duration_ms']['zero_observed_rows'], 1)
        self.assertEqual(statistics['ip_bytes']['proxy_rows'], 1)
        self.assertEqual(statistics['byte_rate_s']['missing_rows'], 1)
        self.assertIsNone(statistics['byte_rate_s']['median'])
        json.dumps(statistics, allow_nan=False)

    def test_saved_audit_rejects_changed_artifacts(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory)
            report = path / 'report.json'
            report.write_text('{"training_started":false}', encoding='utf-8')
            seal = {'fixture': True}
            (path / 'manifest.json').write_text(json.dumps({'status': 'complete', 'seal': seal,
                'artifacts_sha256': {'report.json': sha256_file(report)}}), encoding='utf-8')
            self.assertFalse(verify_saved_audit(path, seal)['training_started'])
            report.write_text('{"training_started":true}', encoding='utf-8')
            with self.assertRaises(ValueError):
                verify_saved_audit(path, seal)

    def cic_volume_frame(self):
        return pd.DataFrame({'Total Fwd Packets': [2, 3], 'Total Backward Packets': [1, 1],
            'Flow Duration': [1000, 0], 'Total Length of Fwd Packets': [20, 40],
            'Total Length of Bwd Packets': [10, 10], 'Fwd Header Length': [40, 60],
            'Bwd Header Length': [20, 20], 'Label': ['BENIGN', 'DoS']})

    def test_cic_flags_are_excluded_without_mutating_raw_mapping(self):
        frame = self.cic_volume_frame()
        for flag in pfs.FLAG_BITS:
            frame[('CWE' if flag == 'cwr' else flag.upper()) + ' Flag Count'] = [0, 1]
        mapped = pfs.convert(frame, 'cicflowmeter')
        before_values, before_quality = mapped['values'].copy(), mapped['quality'].copy()
        for domain in ('CIC2017', 'CIC2018'):
            for view in ('original', 'final'):
                eligible = semantics.apply_semantic_policy(mapped, domain, view)
                for flag in pfs.FLAG_BITS:
                    j = pfs.INDEX['tcp_' + flag]
                    self.assertTrue(np.isnan(eligible['values'][:, j]).all())
                    self.assertTrue((eligible['quality'][:, j] == 0).all())
                self.assertEqual(eligible['quality'][0, pfs.INDEX['packets']], 2)
                self.assertEqual(eligible['quality'][0, pfs.INDEX['ip_bytes']], 1)
        np.testing.assert_allclose(mapped['values'], before_values, equal_nan=True)
        np.testing.assert_array_equal(mapped['quality'], before_quality)
        self.assertEqual(frame['CWE Flag Count'].tolist(), [0, 1])

    def test_nf_time_exclusions_propagate_without_directional_repair(self):
        frame = pd.DataFrame({'IN_PKTS': [2, 2], 'OUT_PKTS': [0, 0],
            'IN_BYTES': [100, 100], 'OUT_BYTES': [0, 0], 'FLOW_DURATION_MILLISECONDS': [4294467, 0],
            'DURATION_IN': [500, 0], 'DURATION_OUT': [0, 0], 'MIN_IP_PKT_LEN': [0, 40], 'TCP_FLAGS': [2, 0]})
        mapped = pfs.convert(frame, 'netflow_v2')
        eligible = semantics.apply_semantic_policy(mapped, 'NF-UNSW-NB15-v2')
        for name in ('duration_ms', 'byte_rate_s', 'packet_rate_s'):
            self.assertTrue(np.isnan(eligible['values'][:, pfs.INDEX[name]]).all())
            self.assertEqual(eligible['capability'][pfs.INDEX[name]], 0)
        self.assertEqual(mapped['values'][0, pfs.INDEX['duration_ms']], 4294467)
        self.assertEqual(mapped['values'][1, pfs.INDEX['duration_ms']], 0)
        self.assertTrue(np.isnan(eligible['values'][0, pfs.INDEX['ip_packet_min_bytes']]))
        self.assertEqual(eligible['values'][1, pfs.INDEX['ip_packet_min_bytes']], 40)
        self.assertEqual(eligible['quality'][1, pfs.INDEX['tcp_syn']], 2)
        self.assertEqual(eligible['values'][1, pfs.INDEX['tcp_syn']], 0)

    def test_ugr_observed_zero_flags_remain_distinct_from_absent_flags(self):
        frame = pd.DataFrame({'ipkt': [2], 'ibyt': [100], 'td': [.05], 'flg': ['......']})
        mapped = pfs.convert(frame, 'ugr16')
        eligible = semantics.apply_semantic_policy(mapped, 'UGR16')
        self.assertEqual(eligible['quality'][0, pfs.INDEX['tcp_syn']], 2)
        self.assertEqual(eligible['values'][0, pfs.INDEX['tcp_syn']], 0)
        self.assertEqual(eligible['quality'][0, pfs.INDEX['tcp_ece']], 0)
        ece = next(f for f in eligible['features'] if f['feature'] == 'tcp_ece')
        self.assertEqual(ece['status'], 'missing')
        self.assertEqual(ece['excluded_observed_rows'], 0)

    def test_policy_rejects_unknown_collection_or_mismatched_exporter(self):
        mapped = pfs.convert(self.cic_volume_frame(), 'cicflowmeter')
        with self.assertRaises(ValueError):
            semantics.apply_semantic_policy(mapped, 'new_unverified_collection')
        with self.assertRaises(ValueError):
            semantics.apply_semantic_policy(mapped, 'UGR16')
        with self.assertRaises(ValueError):
            semantics.apply_semantic_policy(mapped, 'CIC2017', 'unverified')

    def test_disabling_byte_proxy_never_recovers_missing_bytes(self):
        mapped = pfs.convert(self.cic_volume_frame(), 'cicflowmeter', cic_byte_proxy=False)
        eligible = semantics.apply_semantic_policy(mapped, 'CIC2017')
        for name in ('ip_bytes', 'mean_ip_packet_bytes', 'byte_rate_s', 'reverse_byte_share'):
            self.assertEqual(eligible['capability'][pfs.INDEX[name]], 0)
            self.assertTrue(np.isnan(eligible['values'][:, pfs.INDEX[name]]).all())

    def test_binary_only_counts_are_a_signal_not_a_flag_reconstruction(self):
        frame = self.cic_volume_frame()
        frame['SYN Flag Count'] = [0, 1]
        mapped = pfs.convert(frame, 'cicflowmeter')
        checks = semantic_diagnostics(frame, mapped, 'cicflowmeter',
                                     pd.DataFrame({'_pfs_audit_source_byte': [100, 200]}))
        signal = next(c for c in checks if c['check'] == 'cic_syn_binary_only_count_in_multipacket_sample')
        self.assertEqual(signal['flagged_rows'], 2)
        np.testing.assert_array_equal(mapped['values'][:, pfs.INDEX['tcp_syn']], [0, 1])
        frame['SYN Flag Count'] = [0, 2]
        checks = semantic_diagnostics(frame, pfs.convert(frame, 'cicflowmeter'), 'cicflowmeter',
                                     pd.DataFrame({'_pfs_audit_source_byte': [100, 200]}))
        signal = next(c for c in checks if c['check'] == 'cic_syn_binary_only_count_in_multipacket_sample')
        self.assertEqual(signal['flagged_rows'], 0)

    def test_retired_profiles_cannot_start_new_sampling_or_training(self):
        from models.portable_general_experiment import run_pilot
        for profile in semantics.POLICY['retired_training_profiles']:
            with self.assertRaisesRegex(ValueError, 'Profili storici sospesi'):
                semantics.reject_retired_profiles([profile])
            with tempfile.TemporaryDirectory() as directory:
                output = Path(directory) / 'must_not_exist'
                with patch('models.portable_general_experiment.inventory', side_effect=AssertionError('Input letti prima del controllo')):
                    for mode in ('sources', 'pilot'):
                        with self.assertRaisesRegex(ValueError, 'Profili storici sospesi'):
                            run_pilot(Path(directory), output, {'profiles': [profile]}, nf_roles=None, mode=mode)
                self.assertFalse(output.exists())

    def test_published_audit_keeps_mapping_and_eligibility_separate_and_reuses_hashes(self):
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory)
            (root / 'tools').mkdir()
            for name in ('test_portable_general.py', 'test_portable_general_audit.py'):
                (root / 'tools' / name).write_text('# synthetic test identity\n', encoding='utf-8')
            nf = root / 'nf.csv'
            pd.DataFrame({'IN_PKTS': [2], 'OUT_PKTS': [0], 'IN_BYTES': [100], 'OUT_BYTES': [0],
                'FLOW_DURATION_MILLISECONDS': [4294467], 'DURATION_IN': [500], 'DURATION_OUT': [0],
                'TCP_FLAGS': [2], 'Label': [0]}).to_csv(nf, index=False)
            original, final = root / 'cic_original.csv', root / 'cic_final.csv'
            frame = self.cic_volume_frame()
            frame['SYN Flag Count'] = [0, 1]
            frame.to_csv(original, index=False)
            frame.to_csv(final, index=False)
            ugr = root / 'ugr.csv'
            ugr.write_text('2016-08-01 00:10:09,0.05,1.2.3.4,5.6.7.8,1234,443,TCP,.AP.S.,0,0,2,100,background\n', encoding='utf-8')
            records = [
                {'domain': 'NF-UNSW-NB15-v2', 'view': 'original', 'exporter': 'netflow_v2', 'path': nf},
                {'domain': 'CIC2017', 'view': 'original', 'exporter': 'cicflowmeter', 'path': original},
                {'domain': 'CIC2017', 'view': 'final', 'exporter': 'cicflowmeter', 'path': final},
                {'domain': 'UGR16', 'view': 'original', 'exporter': 'ugr16', 'path': ugr}]
            # recovery pairs are keyed by original filename, as in the real inventory.
            final_dir = root / 'final'
            final_dir.mkdir()
            final.rename(final_dir / original.name)
            records[2]['path'] = final_dir / original.name
            with patch('flow_pipeline.portable_general_audit.audit_inventory', return_value=records):
                first = run_information_audit(root, rows_per_window=2, fractions=(0.,), progress=lambda _: None)
                second = run_information_audit(root, rows_per_window=2, fractions=(0.,), progress=lambda _: None)
            self.assertEqual(first, second)
            self.assertFalse(first['ready_for_training'])
            candidates = first['semantic_candidates']
            self.assertEqual(candidates['exact_mapping_candidates_on_every_original_file'], ['packets'])
            self.assertEqual(set(candidates['candidates_allowing_declared_proxies_on_every_original_file']),
                             {'ip_bytes', 'packets', 'mean_ip_packet_bytes'})
            self.assertIsNone(candidates['accepted_training_profile'])
            saved = Path(first['directory'])
            mapped_csv = pd.read_csv(saved / 's' / '00' / 'pfs29.csv')
            eligible_csv = pd.read_csv(saved / 's' / '00' / 'eligible29.csv')
            self.assertEqual(mapped_csv['duration_ms'].iloc[0], 4294467)
            self.assertTrue(eligible_csv['duration_ms'].isna().all())
            self.assertEqual(eligible_csv['_pfs_audit_source_byte'].tolist(), mapped_csv['_pfs_audit_source_byte'].tolist())
            manifest = json.loads((saved / 'manifest.json').read_text(encoding='utf-8'))
            self.assertIn('s/00/eligible29.csv',
                          {name.replace('\\', '/') for name in manifest['artifacts_sha256']})
            self.assertEqual(verify_saved_audit(saved, first['seal']), first)
            self.assertTrue((saved / 'eligibility_matrix.csv').is_file())


def run_audit_fixtures():
    result = unittest.TextTestRunner(verbosity=2).run(
        unittest.defaultTestLoader.loadTestsFromTestCase(PortableGeneralAuditTests))
    if not result.wasSuccessful():
        raise AssertionError('Fixture audit informazioni fallite; nessun audit pubblicato.')
    return {'tests_run': result.testsRun, 'passed': True}


if __name__ == '__main__':
    run_audit_fixtures()
''', 'b24912891933877954c560848b61accf75aac16f3afd2bb37af1d3c54ed797c9')


### tools/test_notebook_standalone.py (incorporato)

In [ ]:
_pfs_put('tools/test_notebook_standalone.py', r'''import hashlib
import tempfile
import unittest
from pathlib import Path
from unittest.mock import patch
import sys

def run_standalone_fixtures(resources, expected, path_type, real_path_type, runtime, notebook_names, notebook_dir):
    class StandaloneTests(unittest.TestCase):
        def test_all_resources_match_their_actual_in_memory_bytes(self):
            for relative, text in resources.items():
                path = path_type(runtime['ROOT'])/relative
                self.assertEqual(hashlib.sha256(path.read_bytes()).hexdigest(), expected[relative])
                with path.open('rb') as handle:
                    self.assertEqual(handle.read(), text.encode('utf-8'))
        def test_source_access_does_not_open_external_project_files(self):
            with patch.object(real_path_type, 'open', side_effect=AssertionError('External file was opened')):
                for relative, text in resources.items():
                    path = path_type(runtime['ROOT'])/relative
                    with path.open('rb') as handle:
                        self.assertEqual(handle.read(), text.encode('utf-8'))
        def test_embedded_sources_cannot_be_overwritten(self):
            for relative in resources:
                with self.assertRaises(PermissionError):
                    (path_type(runtime['ROOT'])/relative).open('w')
        def test_ordinary_data_files_still_use_real_io(self):
            with tempfile.TemporaryDirectory() as directory:
                path = path_type(directory)/'ordinary.csv'
                path.write_text('a,b\n1,2\n', encoding='utf-8')
                self.assertEqual(path.read_text(encoding='utf-8'), 'a,b\n1,2\n')
        def test_project_modules_are_registered_from_embedded_sources(self):
            for relative in resources:
                if relative.endswith('.py') and '/' in relative:
                    module = sys.modules[relative[:-3].replace('/', '.')]
                    self.assertEqual(path_type(module.__file__).read_text(encoding='utf-8'), resources[relative])
        def test_function_inspection_uses_embedded_source_without_file_access(self):
            import inspect
            with patch('linecache.tokenize.open', side_effect=AssertionError('External source inspected')):
                source = inspect.getsource(sys.modules['flow_pipeline.portable_general'].convert)
            self.assertIn(source, resources['flow_pipeline/portable_general.py'])
        def test_shared_code_matches_the_other_final_notebook(self):
            import json
            for name in ('Dataset_NetFlow_Preparation.ipynb', 'Autoencoder_NetFlow_Multidomain.ipynb'):
                path = real_path_type(notebook_dir)/name
                if not path.exists():
                    continue
                nb = json.loads(path.read_text(encoding='utf-8'))
                for relative, digest in nb['metadata']['standalone_release']['shared_resource_sha256'].items():
                    self.assertEqual(expected[relative], digest)
        def test_nf_signatures_preserve_decimal_precision_and_ignore_labels(self):
            if 'notebook_nf_roles' not in sys.modules:
                return
            import numpy as np
            import pandas as pd
            roles = sys.modules['notebook_nf_roles']
            frame = pd.DataFrame({name:['0','0','0'] for name in roles.RAW_ID_COLUMNS})
            frame['IPV4_SRC_ADDR'] = '1.2.3.4'
            frame['IPV4_DST_ADDR'] = '5.6.7.8'
            frame['IN_BYTES'] = ['9007199254740992','9007199254740993','900719925474099200e-2']
            frame['Label'] = ['0','1','1']
            signatures = roles.raw_record_signatures(frame)
            self.assertEqual(signatures[0], signatures[2])
            self.assertNotEqual(signatures[0], signatures[1])
            changed = frame.copy()
            changed['Label'] = ['1','0','0']
            np.testing.assert_array_equal(signatures, roles.raw_record_signatures(changed))
            np.testing.assert_array_equal(roles.signature_roles(signatures), roles.signature_roles(roles.raw_record_signatures(changed)))
    result = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(StandaloneTests))
    if not result.wasSuccessful():
        raise AssertionError('Fixture autonomia notebook fallite: non avviare audit o training.')
    return {'tests_run':result.testsRun, 'passed':True}''', 'ced0a8a93c6d493f1b0b3f2c492a0d020f4c7f2a4eaad41084d13f35b70d2954')


In [ ]:
ROOT = _pfs_initialize(ROOT, NOTEBOOK_RUNTIME_SOURCE)
print('Codice PFS caricato dal notebook; nessun modulo locale esterno necessario.')


## Catalogo, disponibilità e audit congelato
La tabella di disponibilità è una misura strutturale: non certifica tutte le righe o equivalenza fra estrattori. Qualità 0: assente/esclusa; 1: proxy; 2: mapping diretto o derivazione esatta.


In [ ]:
FROZEN_AUDIT_JSON = r'''{"origin":{"report":"D:\\Users\\Anton\\Desktop\\Progetti VsCode\\Antigravity\\Rilevamento di Anomalie del Traffico di Rete con Autoencoder\\Rilevamento-di-Anomalie-del-Traffico-di-Rete-con-Autoencoder\\outputs\\pfsa\\10fa11cd75aae24f\\report.json","sha256":"6c6f30e49074cb0b6f491e2ffb9c20179232228a88f29fd256e71a8d1ab7c03c"},"version":"portable_general_information_audit_v3_semantic_eligibility","domain_summary":[{"domain":"NF-CSE-CIC-IDS2018-v2","view":"original","files":1,"sample_rows":4000,"exact_catalog_on_every_file":29,"proxy_catalog_on_every_file":0,"not_available_on_every_file":0,"invalid_records":0,"benign_sample_rows":3534,"attack_sample_rows":466,"unknown_label_sample_rows":0,"rejected_candidate_lines":0,"semantic_checks_flagged":3},{"domain":"NF-UNSW-NB15-v2","view":"original","files":1,"sample_rows":4000,"exact_catalog_on_every_file":29,"proxy_catalog_on_every_file":0,"not_available_on_every_file":0,"invalid_records":0,"benign_sample_rows":3895,"attack_sample_rows":105,"unknown_label_sample_rows":0,"rejected_candidate_lines":0,"semantic_checks_flagged":3},{"domain":"NF-ToN-IoT-v2","view":"original","files":1,"sample_rows":4000,"exact_catalog_on_every_file":29,"proxy_catalog_on_every_file":0,"not_available_on_every_file":0,"invalid_records":0,"benign_sample_rows":1124,"attack_sample_rows":2876,"unknown_label_sample_rows":0,"rejected_candidate_lines":0,"semantic_checks_flagged":3},{"domain":"NF-BoT-IoT-v2","view":"original","files":1,"sample_rows":4000,"exact_catalog_on_every_file":29,"proxy_catalog_on_every_file":0,"not_available_on_every_file":0,"invalid_records":0,"benign_sample_rows":16,"attack_sample_rows":3984,"unknown_label_sample_rows":0,"rejected_candidate_lines":0,"semantic_checks_flagged":3},{"domain":"CIC2017","view":"original","files":8,"sample_rows":32000,"exact_catalog_on_every_file":11,"proxy_catalog_on_every_file":9,"not_available_on_every_file":9,"invalid_records":1,"benign_sample_rows":26856,"attack_sample_rows":5144,"unknown_label_sample_rows":0,"rejected_candidate_lines":0,"semantic_checks_flagged":89},{"domain":"CIC2017","view":"final","files":8,"sample_rows":32000,"exact_catalog_on_every_file":11,"proxy_catalog_on_every_file":8,"not_available_on_every_file":10,"invalid_records":0,"benign_sample_rows":26791,"attack_sample_rows":5209,"unknown_label_sample_rows":0,"rejected_candidate_lines":0,"semantic_checks_flagged":56},{"domain":"CIC2018","view":"original","files":10,"sample_rows":39999,"exact_catalog_on_every_file":17,"proxy_catalog_on_every_file":9,"not_available_on_every_file":3,"invalid_records":0,"benign_sample_rows":29511,"attack_sample_rows":10488,"unknown_label_sample_rows":0,"rejected_candidate_lines":1,"semantic_checks_flagged":107},{"domain":"CIC2018","view":"final","files":10,"sample_rows":40000,"exact_catalog_on_every_file":11,"proxy_catalog_on_every_file":8,"not_available_on_every_file":10,"invalid_records":0,"benign_sample_rows":29473,"attack_sample_rows":10527,"unknown_label_sample_rows":0,"rejected_candidate_lines":0,"semantic_checks_flagged":72},{"domain":"UGR16","view":"original","files":1,"sample_rows":3998,"exact_catalog_on_every_file":21,"proxy_catalog_on_every_file":0,"not_available_on_every_file":8,"invalid_records":0,"benign_sample_rows":3977,"attack_sample_rows":21,"unknown_label_sample_rows":0,"rejected_candidate_lines":2,"semantic_checks_flagged":1}],"eligibility_summary":[{"domain":"NF-CSE-CIC-IDS2018-v2","view":"original","files":1,"exact_mapping_candidates_on_every_file":26,"proxy_candidates_on_every_file":0,"semantically_excluded_on_at_least_one_file":3,"structurally_missing_on_at_least_one_file":0,"excluded_observed_cells":5269,"notice":"Conteggi del catalogo dopo esclusioni; non tutte le righe sono valide/osservate e non sono informazioni indipendenti."},{"domain":"NF-UNSW-NB15-v2","view":"original","files":1,"exact_mapping_candidates_on_every_file":26,"proxy_candidates_on_every_file":0,"semantically_excluded_on_at_least_one_file":3,"structurally_missing_on_at_least_one_file":0,"excluded_observed_cells":4401,"notice":"Conteggi del catalogo dopo esclusioni; non tutte le righe sono valide/osservate e non sono informazioni indipendenti."},{"domain":"NF-ToN-IoT-v2","view":"original","files":1,"exact_mapping_candidates_on_every_file":26,"proxy_candidates_on_every_file":0,"semantically_excluded_on_at_least_one_file":3,"structurally_missing_on_at_least_one_file":0,"excluded_observed_cells":8053,"notice":"Conteggi del catalogo dopo esclusioni; non tutte le righe sono valide/osservate e non sono informazioni indipendenti."},{"domain":"NF-BoT-IoT-v2","view":"original","files":1,"exact_mapping_candidates_on_every_file":26,"proxy_candidates_on_every_file":0,"semantically_excluded_on_at_least_one_file":3,"structurally_missing_on_at_least_one_file":0,"excluded_observed_cells":14997,"notice":"Conteggi del catalogo dopo esclusioni; non tutte le righe sono valide/osservate e non sono informazioni indipendenti."},{"domain":"CIC2017","view":"original","files":8,"exact_mapping_candidates_on_every_file":4,"proxy_candidates_on_every_file":8,"semantically_excluded_on_at_least_one_file":8,"structurally_missing_on_at_least_one_file":9,"excluded_observed_cells":256000,"notice":"Conteggi del catalogo dopo esclusioni; non tutte le righe sono valide/osservate e non sono informazioni indipendenti."},{"domain":"CIC2017","view":"final","files":8,"exact_mapping_candidates_on_every_file":4,"proxy_candidates_on_every_file":8,"semantically_excluded_on_at_least_one_file":7,"structurally_missing_on_at_least_one_file":10,"excluded_observed_cells":224000,"notice":"Conteggi del catalogo dopo esclusioni; non tutte le righe sono valide/osservate e non sono informazioni indipendenti."},{"domain":"CIC2018","view":"original","files":10,"exact_mapping_candidates_on_every_file":10,"proxy_candidates_on_every_file":8,"semantically_excluded_on_at_least_one_file":8,"structurally_missing_on_at_least_one_file":3,"excluded_observed_cells":319992,"notice":"Conteggi del catalogo dopo esclusioni; non tutte le righe sono valide/osservate e non sono informazioni indipendenti."},{"domain":"CIC2018","view":"final","files":10,"exact_mapping_candidates_on_every_file":4,"proxy_candidates_on_every_file":8,"semantically_excluded_on_at_least_one_file":7,"structurally_missing_on_at_least_one_file":10,"excluded_observed_cells":280000,"notice":"Conteggi del catalogo dopo esclusioni; non tutte le righe sono valide/osservate e non sono informazioni indipendenti."},{"domain":"UGR16","view":"original","files":1,"exact_mapping_candidates_on_every_file":21,"proxy_candidates_on_every_file":0,"semantically_excluded_on_at_least_one_file":0,"structurally_missing_on_at_least_one_file":8,"excluded_observed_cells":0,"notice":"Conteggi del catalogo dopo esclusioni; non tutte le righe sono valide/osservate e non sono informazioni indipendenti."}],"semantic_candidates":{"exact_mapping_candidates_on_every_original_file":["packets"],"candidates_allowing_declared_proxies_on_every_original_file":["ip_bytes","packets","mean_ip_packet_bytes"],"exploratory_common_volume_coordinates":["packets","ip_bytes","mean_ip_packet_bytes"],"exploratory_volume_complete_in_samples":true,"exploratory_volume_notice":"Three coordinates from two counters. CIC IP bytes are estimated, so this is not an exact universal core or a selected training profile.","all29_exact_eligible_on_every_original_file":false,"accepted_training_profile":null,"ready_for_training":false,"next_decision":"Valutare un PFS a nucleo ridotto con estensioni dichiarate, oppure CSV CIC riestratti con tool verificato. Nessun riempimento artificiale fino a 29.","notice":"Eligibility is a conservative review decision, not a certification of physical accuracy, session equivalence or predictive importance. Mapping quality is preserved separately."},"sample_count":41,"notice":"Risultati congelati precedenti alla riorganizzazione del codice. Audit limitato a campioni."}'''
FROZEN_AUDIT = json.loads(FROZEN_AUDIT_JSON)


In [ ]:
from flow_pipeline import portable_general as pfs
from flow_pipeline import portable_general_semantics as semantics
from flow_pipeline.portable_general_audit import run_information_audit
from tools.test_portable_general import run_fixtures
from tools.test_portable_general_audit import run_audit_fixtures
from tools.test_notebook_standalone import run_standalone_fixtures

if DATASET_PFS_MODE not in {'report', 'test', 'audit'}:
    raise ValueError('Usare report, test o audit.')
show_table(pfs.CONTRACT['features'], 'Catalogo PFS: 29 coordinate semantiche')
if DATASET_PFS_MODE == 'report':
    print(FROZEN_AUDIT['notice'])
    show_table(FROZEN_AUDIT['eligibility_summary'], 'Disponibilita dopo le esclusioni semantiche: audit congelato')
    display(FROZEN_AUDIT['semantic_candidates'])
else:
    bundle_checks = _pfs_check_bundle(NOTEBOOK_NAMES)
    standalone_tests = run_standalone_fixtures(_PFS_TEXT, _PFS_HASHES, _NotebookPath, _RealPath,
                                              globals(), NOTEBOOK_NAMES, NOTEBOOK_DIR)
    fixture_results = {'converter': run_fixtures(), 'information_audit': run_audit_fixtures()}
    print('STANDALONE DATASET TEST COMPLETATO', bundle_checks, standalone_tests, fixture_results)
    if DATASET_PFS_MODE == 'audit':
        report = run_information_audit(ROOT, rows_per_window=1000, fractions=(0., .25, .50, .75),
                                       cic_byte_proxy=True, fixtures=fixture_results)
        show_table(report['eligibility_summary'], 'Nuovo audit del codice incorporato')
        print('AUDIT COMPLETATO:', report['directory'])
        print('Inviare il report prima di avviare un nuovo pilot.')